In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import zipfile
import pandas as pd
import json

DATA_DIR = Path(
    "/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet"
)

EXPECTED_FILES = [
    "NF-BoT-IoT-v3.zip",
    "NF-CSE-CIC-IDS2018-v3.zip",
    "NF-ToN-IoT-v3.zip",
    "NF-UNSW-NB15-v3.zip",
]

audit_results = {}

print("=" * 70)
print("PAPER 3 — DATASET INVENTORY AND HEADER AUDIT")
print("=" * 70)

for filename in EXPECTED_FILES:
    zip_path = DATA_DIR / filename

    if not zip_path.exists():
        print(f"\nMISSING: {filename}")
        audit_results[filename] = {"status": "missing"}
        continue

    print(f"\nFOUND: {filename}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**3):.3f} GB")

    try:
        with zipfile.ZipFile(zip_path, "r") as archive:
            bad_file = archive.testzip()

            if bad_file is not None:
                print(f"CORRUPTED MEMBER: {bad_file}")
                audit_results[filename] = {
                    "status": "corrupted",
                    "bad_member": bad_file,
                }
                continue

            csv_members = [
                item for item in archive.namelist()
                if item.lower().endswith(".csv")
                and not item.startswith("__MACOSX")
            ]

            print(f"CSV members: {len(csv_members)}")

            member_results = []

            for member in csv_members:
                info = archive.getinfo(member)

                print(f"\n  Member: {member}")
                print(
                    f"  Uncompressed size: "
                    f"{info.file_size / (1024**3):.3f} GB"
                )

                with archive.open(member) as csv_file:
                    sample = pd.read_csv(
                        csv_file,
                        nrows=5,
                        low_memory=False,
                    )

                columns = sample.columns.tolist()

                timestamp_columns = [
                    column for column in columns
                    if "TIME" in column.upper()
                    or "DATE" in column.upper()
                ]

                label_columns = [
                    column for column in columns
                    if column.upper() in {
                        "LABEL",
                        "ATTACK",
                        "BINARY_LABEL",
                        "ATTACK_LABEL",
                        "CLASS",
                    }
                ]

                print(f"  Number of columns: {len(columns)}")
                print(f"  Timestamp candidates: {timestamp_columns}")
                print(f"  Label candidates: {label_columns}")
                print(f"  Columns: {columns}")

                member_results.append({
                    "member": member,
                    "uncompressed_bytes": info.file_size,
                    "number_of_columns": len(columns),
                    "columns": columns,
                    "timestamp_candidates": timestamp_columns,
                    "label_candidates": label_columns,
                })

            audit_results[filename] = {
                "status": "pass",
                "zip_bytes": zip_path.stat().st_size,
                "members": member_results,
            }

    except Exception as error:
        print(f"ERROR: {type(error).__name__}: {error}")
        audit_results[filename] = {
            "status": "error",
            "error": str(error),
        }

AUDIT_DIR = DATA_DIR / "Paper3_Audit"
AUDIT_DIR.mkdir(exist_ok=True)

output_file = AUDIT_DIR / "dataset_inventory_audit.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(audit_results, file, indent=2)

print("\n" + "=" * 70)
print(f"AUDIT SAVED: {output_file}")
print("=" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER 3 — DATASET INVENTORY AND HEADER AUDIT

FOUND: NF-BoT-IoT-v3.zip
ZIP size: 0.068 GB
CSV members: 2

  Member: d509c9db7490cf92_NFV3DATA-A11964_A11964/data/NF-BoT-IoT-v3.csv
  Uncompressed size: 3.555 GB
  Number of columns: 55
  Timestamp candidates: []
  Label candidates: ['Label', 'Attack']
  Columns: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 

In [3]:
from pathlib import Path
from collections import Counter, defaultdict
import zipfile
import pandas as pd
import json
import time

DATA_DIR = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

AUDIT_DIR = DATA_DIR / "Paper3_Audit" / "temporal_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "NF_BoT_IoT_v3": "NF-BoT-IoT-v3.zip",
    "NF_CSE_CIC_IDS2018_v3": "NF-CSE-CIC-IDS2018-v3.zip",
    "NF_ToN_IoT_v3": "NF-ToN-IoT-v3.zip",
    "NF_UNSW_NB15_v3": "NF-UNSW-NB15-v3.zip",
}

USE_COLUMNS = [
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "Label",
    "Attack",
]

CHUNK_SIZE = 500_000

# Prespecified before examining temporal counts
BLOCK_HOURS = 6
MIN_BENIGN_PER_PERIOD = 2_000
MIN_ATTACK_PER_PERIOD = 2_000


def update_period_counter(counter, periods, labels):
    frame = pd.DataFrame({
        "period": periods,
        "label": labels,
    }).dropna()

    frame = frame[frame["label"].isin([0, 1])]

    if frame.empty:
        return

    grouped = frame.groupby(["period", "label"]).size()

    for (period, label), count in grouped.items():
        counter[str(period)][int(label)] += int(count)


def period_table(counter, period_type):
    rows = []

    for period in sorted(counter):
        benign = int(counter[period].get(0, 0))
        attack = int(counter[period].get(1, 0))

        eligible = (
            benign >= MIN_BENIGN_PER_PERIOD
            and attack >= MIN_ATTACK_PER_PERIOD
        )

        rows.append({
            "period_type": period_type,
            "period": period,
            "benign": benign,
            "attack": attack,
            "total": benign + attack,
            "eligible": eligible,
        })

    return pd.DataFrame(rows)


all_summaries = []

print("=" * 78)
print("PAPER 3 — STREAMING TEMPORAL AND LABEL AUDIT")
print("=" * 78)

for dataset_name, zip_filename in DATASETS.items():

    checkpoint = AUDIT_DIR / f"{dataset_name}_audit.json"

    # Resume support
    if checkpoint.exists():
        with open(checkpoint, "r", encoding="utf-8") as file:
            existing = json.load(file)

        all_summaries.append(existing)
        print(f"\nSKIPPED—checkpoint exists: {dataset_name}")
        continue

    zip_path = DATA_DIR / zip_filename

    print(f"\n{'-' * 78}")
    print(f"DATASET: {dataset_name}")
    print(f"ZIP: {zip_path}")
    print(f"{'-' * 78}")

    if not zip_path.exists():
        print("ERROR: ZIP file not found.")
        continue

    row_count = 0
    invalid_start_timestamp = 0
    invalid_end_timestamp = 0
    invalid_label_count = 0

    label_counts = Counter()
    attack_counts = Counter()
    daily_counts = defaultdict(Counter)
    block_counts = defaultdict(Counter)

    minimum_time = None
    maximum_time = None

    start_time = time.time()

    with zipfile.ZipFile(zip_path, "r") as archive:

        candidate_members = [
            member for member in archive.namelist()
            if member.lower().endswith(".csv")
            and "netflow_v3_features" not in member.lower()
            and not member.startswith("__MACOSX")
        ]

        if not candidate_members:
            raise RuntimeError(
                f"No principal dataset CSV found in {zip_filename}"
            )

        # Select the largest CSV as the principal dataset
        csv_member = max(
            candidate_members,
            key=lambda member: archive.getinfo(member).file_size,
        )

        print(f"CSV member: {csv_member}")

        with archive.open(csv_member) as csv_file:

            reader = pd.read_csv(
                csv_file,
                usecols=USE_COLUMNS,
                chunksize=CHUNK_SIZE,
                low_memory=False,
            )

            for chunk_number, chunk in enumerate(reader, start=1):

                rows_in_chunk = len(chunk)
                row_count += rows_in_chunk

                start_numeric = pd.to_numeric(
                    chunk["FLOW_START_MILLISECONDS"],
                    errors="coerce",
                )

                end_numeric = pd.to_numeric(
                    chunk["FLOW_END_MILLISECONDS"],
                    errors="coerce",
                )

                labels = pd.to_numeric(
                    chunk["Label"],
                    errors="coerce",
                )

                start_datetime = pd.to_datetime(
                    start_numeric,
                    unit="ms",
                    utc=True,
                    errors="coerce",
                )

                end_datetime = pd.to_datetime(
                    end_numeric,
                    unit="ms",
                    utc=True,
                    errors="coerce",
                )

                invalid_start_timestamp += int(
                    start_datetime.isna().sum()
                )

                invalid_end_timestamp += int(
                    end_datetime.isna().sum()
                )

                invalid_label_count += int(
                    (~labels.isin([0, 1])).sum()
                )

                valid_labels = labels[labels.isin([0, 1])].astype(int)

                label_counts.update(
                    valid_labels.value_counts().to_dict()
                )

                attack_values = (
                    chunk["Attack"]
                    .fillna("__MISSING__")
                    .astype(str)
                    .str.strip()
                )

                attack_counts.update(
                    attack_values.value_counts().to_dict()
                )

                valid_times = start_datetime.dropna()

                if not valid_times.empty:
                    chunk_minimum = valid_times.min()
                    chunk_maximum = valid_times.max()

                    if (
                        minimum_time is None
                        or chunk_minimum < minimum_time
                    ):
                        minimum_time = chunk_minimum

                    if (
                        maximum_time is None
                        or chunk_maximum > maximum_time
                    ):
                        maximum_time = chunk_maximum

                days = start_datetime.dt.strftime("%Y-%m-%d")

                blocks = (
                    start_datetime
                    .dt.floor(f"{BLOCK_HOURS}h")
                    .dt.strftime("%Y-%m-%d %H:%M:%S")
                )

                update_period_counter(
                    daily_counts,
                    days,
                    labels,
                )

                update_period_counter(
                    block_counts,
                    blocks,
                    labels,
                )

                if chunk_number % 5 == 0:
                    elapsed_minutes = (
                        time.time() - start_time
                    ) / 60

                    print(
                        f"  Chunks: {chunk_number:,} | "
                        f"Rows: {row_count:,} | "
                        f"Elapsed: {elapsed_minutes:.1f} min"
                    )

    daily_table = period_table(
        daily_counts,
        "calendar_day",
    )

    block_table = period_table(
        block_counts,
        f"{BLOCK_HOURS}_hour_block",
    )

    daily_file = AUDIT_DIR / f"{dataset_name}_daily_counts.csv"
    block_file = AUDIT_DIR / f"{dataset_name}_6hour_counts.csv"

    daily_table.to_csv(daily_file, index=False)
    block_table.to_csv(block_file, index=False)

    elapsed_minutes = (time.time() - start_time) / 60

    summary = {
        "dataset": dataset_name,
        "zip_file": zip_filename,
        "csv_member": csv_member,
        "total_rows": int(row_count),
        "benign_label_0": int(label_counts.get(0, 0)),
        "attack_label_1": int(label_counts.get(1, 0)),
        "invalid_labels": int(invalid_label_count),
        "invalid_start_timestamps": int(
            invalid_start_timestamp
        ),
        "invalid_end_timestamps": int(
            invalid_end_timestamp
        ),
        "minimum_start_time": (
            minimum_time.isoformat()
            if minimum_time is not None else None
        ),
        "maximum_start_time": (
            maximum_time.isoformat()
            if maximum_time is not None else None
        ),
        "calendar_days": int(len(daily_table)),
        "eligible_calendar_days": int(
            daily_table["eligible"].sum()
            if not daily_table.empty else 0
        ),
        "six_hour_blocks": int(len(block_table)),
        "eligible_six_hour_blocks": int(
            block_table["eligible"].sum()
            if not block_table.empty else 0
        ),
        "attack_category_counts": {
            str(key): int(value)
            for key, value in attack_counts.items()
        },
        "elapsed_minutes": round(elapsed_minutes, 2),
    }

    with open(checkpoint, "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)

    all_summaries.append(summary)

    print(f"\nCOMPLETED: {dataset_name}")
    print(f"Rows: {row_count:,}")
    print(f"Label 0—benign: {label_counts.get(0, 0):,}")
    print(f"Label 1—attack: {label_counts.get(1, 0):,}")
    print(f"Calendar days: {len(daily_table)}")
    print(
        "Eligible calendar days: "
        f"{daily_table['eligible'].sum()}"
    )
    print(f"Six-hour blocks: {len(block_table)}")
    print(
        "Eligible six-hour blocks: "
        f"{block_table['eligible'].sum()}"
    )
    print(f"Elapsed: {elapsed_minutes:.1f} minutes")


summary_table = pd.DataFrame([
    {
        "dataset": item["dataset"],
        "total_rows": item["total_rows"],
        "benign": item["benign_label_0"],
        "attack": item["attack_label_1"],
        "invalid_labels": item["invalid_labels"],
        "invalid_start_times": item[
            "invalid_start_timestamps"
        ],
        "calendar_days": item["calendar_days"],
        "eligible_days": item["eligible_calendar_days"],
        "six_hour_blocks": item["six_hour_blocks"],
        "eligible_6h_blocks": item[
            "eligible_six_hour_blocks"
        ],
    }
    for item in all_summaries
])

summary_path = AUDIT_DIR / "combined_temporal_audit.csv"
summary_table.to_csv(summary_path, index=False)

print("\n" + "=" * 78)
print("COMBINED TEMPORAL AUDIT")
print("=" * 78)
print(summary_table.to_string(index=False))
print(f"\nSaved: {summary_path}")

PAPER 3 — STREAMING TEMPORAL AND LABEL AUDIT

------------------------------------------------------------------------------
DATASET: NF_BoT_IoT_v3
ZIP: /content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet/NF-BoT-IoT-v3.zip
------------------------------------------------------------------------------
CSV member: d509c9db7490cf92_NFV3DATA-A11964_A11964/data/NF-BoT-IoT-v3.csv
  Chunks: 5 | Rows: 2,500,000 | Elapsed: 0.9 min
  Chunks: 10 | Rows: 5,000,000 | Elapsed: 1.7 min
  Chunks: 15 | Rows: 7,500,000 | Elapsed: 2.5 min
  Chunks: 20 | Rows: 10,000,000 | Elapsed: 3.3 min
  Chunks: 25 | Rows: 12,500,000 | Elapsed: 4.2 min
  Chunks: 30 | Rows: 15,000,000 | Elapsed: 5.0 min

COMPLETED: NF_BoT_IoT_v3
Rows: 16,933,808
Label 0—benign: 51,989
Label 1—attack: 16,881,819
Calendar days: 6
Eligible calendar days: 4
Six-hour blocks: 10
Eligible six-hour blocks: 5
Elapsed: 5.6 minutes

------------------------------------------------------------------------------
DATASET: NF_CS

In [4]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

PROTOCOL_DIR = ROOT / "Paper3_Protocol"
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)

protocol = {
    "protocol_id": "P3_FEATURE_STABILITY_V1",
    "status": "FROZEN_BEFORE_MODELLING",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "working_title": (
        "Does Source-Side Feature Stability Warn of "
        "Cross-Dataset NIDS Failure?"
    ),

    "primary_question": (
        "Does feature-importance stability across genuine source "
        "temporal environments predict cross-dataset failure better "
        "than stability across random source samples?"
    ),

    "datasets": {
        "NF_BoT_IoT_v3": {
            "rows": 16933808,
            "eligible_6h_blocks": 5,
        },
        "NF_CSE_CIC_IDS2018_v3": {
            "rows": 20115529,
            "eligible_6h_blocks": 13,
        },
        "NF_ToN_IoT_v3": {
            "rows": 27520260,
            "eligible_6h_blocks": 21,
        },
        "NF_UNSW_NB15_v3": {
            "rows": 2365424,
            "eligible_6h_blocks": 4,
        },
    },

    "task": {
        "type": "binary_classification",
        "benign_label": 0,
        "attack_label": 1,
        "transfer_directions": 12,
    },

    "environment_definition": {
        "timestamp": "FLOW_START_MILLISECONDS",
        "timezone": "UTC",
        "primary_unit": "non_overlapping_6_hour_block",
        "block_operation": "floor timestamp to 6-hour boundary",
        "minimum_benign_per_block": 2000,
        "minimum_attack_per_block": 2000,
        "included_blocks": "all eligible blocks",
        "total_eligible_blocks": 43,
        "calendar_day_analysis": "secondary_sensitivity",
    },

    "features": {
        "available_predictors": 53,
        "excluded_from_models": [
            "FLOW_START_MILLISECONDS",
            "FLOW_END_MILLISECONDS",
            "IPV4_SRC_ADDR",
            "IPV4_DST_ADDR",
        ],
        "primary_predictor_count": 49,
        "timestamp_usage": "environment_definition_only",
        "ip_address_usage": "excluded_completely",
        "label_columns": [
            "Label",
            "Attack",
        ],
    },

    "sampling": {
        "environment_sample_per_block": {
            "benign": 2000,
            "attack": 2000,
            "replacement": False,
        },
        "sampling_stability_control": (
            "Use the same observations as the environmental samples, "
            "but randomly redistribute observations within each class "
            "into matched pseudo-blocks."
        ),
        "source_training_per_class": 25000,
        "source_calibration_per_class": 5000,
        "source_test_per_class": 5000,
        "sampling_method": "deterministic_seeded_without_replacement",
        "seeds": [2027, 2028, 2029, 2030, 2031],
    },

    "preprocessing": {
        "fit_using": "source_training_only",
        "infinite_values": "convert_to_missing",
        "imputation": "source_training_median",
        "logistic_regression_scaling": "StandardScaler",
        "tree_model_scaling": "none",
        "target_adaptation": False,
        "target_imputation_refitting": False,
        "target_scaling_refitting": False,
    },

    "models": {
        "LogisticRegression": {
            "solver": "liblinear",
            "max_iter": 2000,
            "class_weight": None,
        },
        "RandomForest": {
            "n_estimators": 300,
            "max_features": "sqrt",
            "min_samples_leaf": 2,
            "n_jobs": -1,
        },
        "XGBoost": {
            "n_estimators": 600,
            "learning_rate": 0.05,
            "max_depth": 6,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
        },
    },

    "threshold": {
        "selection_data": "source_calibration_only",
        "rule": "maximize balanced accuracy",
        "tie_break": "threshold closest to 0.5",
        "target_recalibration": False,
    },

    "feature_importance": {
        "primary_method": "permutation_importance",
        "scoring": "balanced_accuracy",
        "permutation_repeats": 3,
        "primary_stability_metric": (
            "mean pairwise Spearman rank correlation"
        ),
        "secondary_stability_metric": "top_10_Jaccard",
        "shap_usage": "XGBoost verification only",
    },

    "warning_scores": {
        "environmental_instability": "1 - environmental_stability",
        "false_reassurance_gap": (
            "sampling_stability - environmental_stability"
        ),
        "interpretation": (
            "Higher values indicate greater source-only warning risk."
        ),
    },

    "outcomes": {
        "primary_transfer_failure": (
            "source balanced accuracy - target balanced accuracy"
        ),
        "secondary_metrics": [
            "Macro F1",
            "MCC",
            "PR-AUC",
        ],
        "target_threshold": (
            "Use frozen source-calibrated threshold without adaptation"
        ),
    },

    "confirmatory_rules": {
        "all_source_warning_scores_frozen_before_target_evaluation": True,
        "all_source_models_frozen_before_target_evaluation": True,
        "target_features_not_used_for_warning_construction": True,
        "target_labels_not_used_for_warning_construction": True,
        "negative_and_null_results_reported": True,
    },

    "scope_separation": {
        "semantic_feature_group_masking": False,
        "artificial_feature_loss": False,
        "selective_classification": False,
        "target_domain_adaptation": False,
        "new_deep_learning_architecture": False,
    },

    "robustness_checks": [
        "calendar-day stability where sufficient",
        "top-10 Jaccard stability",
        "XGBoost SHAP verification",
        "exact-feature-duplicate sensitivity audit",
    ],
}

protocol_text = json.dumps(
    protocol,
    indent=2,
    sort_keys=True,
    ensure_ascii=False,
)

protocol_file = (
    PROTOCOL_DIR / "paper3_protocol_frozen_v1.json"
)

protocol_file.write_text(
    protocol_text,
    encoding="utf-8",
)

protocol_hash = hashlib.sha256(
    protocol_text.encode("utf-8")
).hexdigest()

hash_file = (
    PROTOCOL_DIR / "paper3_protocol_frozen_v1.sha256"
)

hash_file.write_text(
    protocol_hash + "\n",
    encoding="utf-8",
)

# Verification
saved_text = protocol_file.read_text(encoding="utf-8")
verified_hash = hashlib.sha256(
    saved_text.encode("utf-8")
).hexdigest()

print("=" * 72)
print("PAPER 3 PROTOCOL FREEZE")
print("=" * 72)
print(f"Protocol: {protocol_file}")
print(f"SHA-256: {protocol_hash}")
print(
    "Verification:",
    "PASS" if verified_hash == protocol_hash else "FAIL",
)
print("Status: FROZEN BEFORE MODELLING")

PAPER 3 PROTOCOL FREEZE
Protocol: /content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet/Paper3_Protocol/paper3_protocol_frozen_v1.json
SHA-256: e39a5007a7f2cbdc348d5179ce593af5766bc6688f5baa3a46a56399f683a488
Verification: PASS
Status: FROZEN BEFORE MODELLING


In [5]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import zipfile
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

AUDIT_DIR = ROOT / "Paper3_Audit" / "temporal_audit"
PROTOCOL_DIR = ROOT / "Paper3_Protocol"
PARTITION_DIR = ROOT / "Paper3_Partitions"
PARTITION_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL_FILE = (
    PROTOCOL_DIR / "paper3_protocol_frozen_v1.json"
)

EXPECTED_PROTOCOL_HASH = (
    "e39a5007a7f2cbdc348d5179ce593af"
    "5766bc6688f5baa3a46a56399f683a488"
)

DATASETS = {
    "NF_BoT_IoT_v3": "NF-BoT-IoT-v3.zip",
    "NF_CSE_CIC_IDS2018_v3": "NF-CSE-CIC-IDS2018-v3.zip",
    "NF_ToN_IoT_v3": "NF-ToN-IoT-v3.zip",
    "NF_UNSW_NB15_v3": "NF-UNSW-NB15-v3.zip",
}

SEEDS = [2027, 2028, 2029, 2030, 2031]

CHUNK_SIZE = 500_000
BLOCK_MS = 6 * 60 * 60 * 1000

ENVIRONMENT_PER_CLASS = 2_000
TRAIN_PER_CLASS = 25_000
CALIBRATION_PER_CLASS = 5_000
SOURCE_TEST_PER_CLASS = 5_000

GLOBAL_REQUIRED = (
    TRAIN_PER_CLASS
    + CALIBRATION_PER_CLASS
    + SOURCE_TEST_PER_CLASS
)

SALT_ENV = np.uint64(0xA24BAED4963EE407)
SALT_GLOBAL = np.uint64(0x9FB21C651E98DF25)
SALT_PSEUDO = np.uint64(0xC13FA9A902A6328F)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for part in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(part)

    return digest.hexdigest()


def splitmix64(values):
    values = values.astype(np.uint64, copy=False)

    with np.errstate(over="ignore"):
        values = values + np.uint64(0x9E3779B97F4A7C15)
        values = (
            values ^ np.right_shift(values, 30)
        ) * np.uint64(0xBF58476D1CE4E5B9)
        values = (
            values ^ np.right_shift(values, 27)
        ) * np.uint64(0x94D049BB133111EB)
        values = values ^ np.right_shift(values, 31)

    return values


def priorities(row_ids, seed, salt):
    values = (
        row_ids.astype(np.uint64, copy=False)
        ^ np.uint64(seed)
        ^ salt
    )
    return splitmix64(values)


def merge_smallest(existing, new_priorities, new_ids, keep):
    if len(new_ids) == 0:
        return existing

    new_priorities = np.asarray(
        new_priorities,
        dtype=np.uint64,
    )
    new_ids = np.asarray(new_ids, dtype=np.int64)

    if existing is None:
        combined_priorities = new_priorities
        combined_ids = new_ids
    else:
        combined_priorities = np.concatenate([
            existing[0],
            new_priorities,
        ])
        combined_ids = np.concatenate([
            existing[1],
            new_ids,
        ])

    if len(combined_ids) > keep:
        selected = np.argpartition(
            combined_priorities,
            keep - 1,
        )[:keep]

        combined_priorities = combined_priorities[selected]
        combined_ids = combined_ids[selected]

    return combined_priorities, combined_ids


def sorted_smallest(item, keep):
    priority_values, row_ids = item

    order = np.lexsort((
        row_ids,
        priority_values,
    ))

    order = order[:keep]

    return (
        priority_values[order],
        row_ids[order],
    )


# Verify that the frozen protocol has not changed
actual_protocol_hash = sha256_file(PROTOCOL_FILE)

print("=" * 78)
print("PAPER 3 — DETERMINISTIC PARTITION-INDEX CONSTRUCTION")
print("=" * 78)
print(f"Expected protocol hash: {EXPECTED_PROTOCOL_HASH}")
print(f"Observed protocol hash: {actual_protocol_hash}")

if actual_protocol_hash != EXPECTED_PROTOCOL_HASH:
    raise RuntimeError(
        "Protocol hash mismatch. Stop: the frozen protocol "
        "may have been modified."
    )

print("Protocol verification: PASS")


for dataset_name, zip_filename in DATASETS.items():

    dataset_output = PARTITION_DIR / dataset_name
    dataset_output.mkdir(parents=True, exist_ok=True)

    completion_file = (
        dataset_output / "partition_index_complete.json"
    )

    if completion_file.exists():
        print(f"\nSKIPPED—already completed: {dataset_name}")
        continue

    block_count_file = (
        AUDIT_DIR / f"{dataset_name}_6hour_counts.csv"
    )

    block_table = pd.read_csv(block_count_file)
    eligible_table = block_table[
        block_table["eligible"].astype(bool)
    ].copy()

    eligible_table["block_datetime"] = pd.to_datetime(
        eligible_table["period"],
        utc=True,
    )

    eligible_table["block_id_ms"] = (
        eligible_table["block_datetime"].astype("int64")
        // 1_000_000
    ).astype("int64")

    eligible_blocks = sorted(
        eligible_table["block_id_ms"].unique().tolist()
    )

    eligible_keys = {
        (int(block_id), int(label))
        for block_id in eligible_blocks
        for label in [0, 1]
    }

    number_of_blocks = len(eligible_blocks)

    # Extra candidates guarantee that enough global rows remain
    # after all selected environment rows are excluded.
    global_candidate_limit = (
        GLOBAL_REQUIRED
        + number_of_blocks * ENVIRONMENT_PER_CLASS
    )

    env_top = {
        seed: {} for seed in SEEDS
    }

    global_top = {
        seed: {
            0: None,
            1: None,
        }
        for seed in SEEDS
    }

    zip_path = ROOT / zip_filename
    processed_rows = 0
    start_time = time.time()

    print(f"\n{'-' * 78}")
    print(f"DATASET: {dataset_name}")
    print(f"Eligible blocks: {number_of_blocks}")
    print(
        "Global candidate limit per class: "
        f"{global_candidate_limit:,}"
    )
    print(f"{'-' * 78}")

    with zipfile.ZipFile(zip_path, "r") as archive:

        members = [
            member for member in archive.namelist()
            if member.lower().endswith(".csv")
            and "netflow_v3_features" not in member.lower()
            and not member.startswith("__MACOSX")
        ]

        csv_member = max(
            members,
            key=lambda member: archive.getinfo(member).file_size,
        )

        print(f"CSV member: {csv_member}")

        with archive.open(csv_member) as csv_file:

            reader = pd.read_csv(
                csv_file,
                usecols=[
                    "FLOW_START_MILLISECONDS",
                    "Label",
                ],
                chunksize=CHUNK_SIZE,
                low_memory=False,
            )

            for chunk_number, chunk in enumerate(
                reader,
                start=1,
            ):
                chunk_length = len(chunk)

                row_ids = np.arange(
                    processed_rows,
                    processed_rows + chunk_length,
                    dtype=np.int64,
                )

                start_values = pd.to_numeric(
                    chunk["FLOW_START_MILLISECONDS"],
                    errors="coerce",
                ).fillna(-1).astype("int64").to_numpy()

                labels = pd.to_numeric(
                    chunk["Label"],
                    errors="coerce",
                ).fillna(-1).astype("int8").to_numpy()

                block_ids = (
                    start_values // BLOCK_MS
                ) * BLOCK_MS

                eligible_mask = (
                    np.isin(block_ids, eligible_blocks)
                    & np.isin(labels, [0, 1])
                )

                eligible_positions = np.flatnonzero(
                    eligible_mask
                )

                grouped_positions = {}

                if len(eligible_positions) > 0:
                    grouping_frame = pd.DataFrame({
                        "position": eligible_positions,
                        "block": block_ids[
                            eligible_positions
                        ],
                        "label": labels[
                            eligible_positions
                        ],
                    })

                    for key, group in grouping_frame.groupby(
                        ["block", "label"],
                        sort=False,
                    ):
                        normalized_key = (
                            int(key[0]),
                            int(key[1]),
                        )

                        if normalized_key in eligible_keys:
                            grouped_positions[
                                normalized_key
                            ] = group[
                                "position"
                            ].to_numpy()

                for seed in SEEDS:
                    env_priorities = priorities(
                        row_ids,
                        seed,
                        SALT_ENV,
                    )

                    global_priorities = priorities(
                        row_ids,
                        seed,
                        SALT_GLOBAL,
                    )

                    for key, positions in grouped_positions.items():
                        existing = env_top[seed].get(key)

                        env_top[seed][key] = merge_smallest(
                            existing=existing,
                            new_priorities=env_priorities[
                                positions
                            ],
                            new_ids=row_ids[positions],
                            keep=ENVIRONMENT_PER_CLASS,
                        )

                    for label in [0, 1]:
                        positions = np.flatnonzero(
                            labels == label
                        )

                        global_top[seed][label] = merge_smallest(
                            existing=global_top[seed][label],
                            new_priorities=global_priorities[
                                positions
                            ],
                            new_ids=row_ids[positions],
                            keep=global_candidate_limit,
                        )

                processed_rows += chunk_length

                if chunk_number % 10 == 0:
                    elapsed = (
                        time.time() - start_time
                    ) / 60

                    print(
                        f"  Chunks: {chunk_number:,} | "
                        f"Rows: {processed_rows:,} | "
                        f"Elapsed: {elapsed:.1f} min"
                    )

    seed_manifests = {}

    for seed in SEEDS:
        seed_directory = dataset_output / f"seed_{seed}"
        seed_directory.mkdir(parents=True, exist_ok=True)

        environment_frames = []

        for block_id in eligible_blocks:
            for label in [0, 1]:
                key = (int(block_id), label)

                if key not in env_top[seed]:
                    raise RuntimeError(
                        f"Missing environment group: "
                        f"{dataset_name}, seed={seed}, key={key}"
                    )

                _, selected_ids = sorted_smallest(
                    env_top[seed][key],
                    ENVIRONMENT_PER_CLASS,
                )

                if len(selected_ids) != ENVIRONMENT_PER_CLASS:
                    raise RuntimeError(
                        f"Insufficient environment rows for {key}"
                    )

                environment_frames.append(
                    pd.DataFrame({
                        "row_id": selected_ids,
                        "Label": label,
                        "block_id_ms": int(block_id),
                    })
                )

        environment_index = pd.concat(
            environment_frames,
            ignore_index=True,
        )

        environment_row_ids = (
            environment_index["row_id"]
            .astype("int64")
            .to_numpy()
        )

        # Create matched random pseudo-environments using exactly
        # the same observations as the temporal environments.
        environment_index[
            "pseudo_environment_id"
        ] = -1

        for label in [0, 1]:
            label_positions = np.flatnonzero(
                environment_index["Label"].to_numpy()
                == label
            )

            label_row_ids = (
                environment_index.iloc[
                    label_positions
                ]["row_id"]
                .astype("int64")
                .to_numpy()
            )

            pseudo_priorities = priorities(
                label_row_ids,
                seed,
                SALT_PSEUDO,
            )

            pseudo_order = np.lexsort((
                label_row_ids,
                pseudo_priorities,
            ))

            ordered_positions = label_positions[
                pseudo_order
            ]

            pseudo_values = np.repeat(
                np.arange(number_of_blocks),
                ENVIRONMENT_PER_CLASS,
            )

            environment_index.loc[
                ordered_positions,
                "pseudo_environment_id",
            ] = pseudo_values

        environment_index[
            "actual_environment_id"
        ] = pd.to_datetime(
            environment_index["block_id_ms"],
            unit="ms",
            utc=True,
        ).dt.strftime("%Y-%m-%dT%H:%M:%SZ")

        # Select disjoint train/calibration/source-test rows.
        global_frames = []

        for label in [0, 1]:
            priority_values, candidate_ids = (
                global_top[seed][label]
            )

            not_environment = ~np.isin(
                candidate_ids,
                environment_row_ids,
            )

            priority_values = priority_values[
                not_environment
            ]
            candidate_ids = candidate_ids[
                not_environment
            ]

            priority_values, candidate_ids = sorted_smallest(
                (priority_values, candidate_ids),
                GLOBAL_REQUIRED,
            )

            if len(candidate_ids) < GLOBAL_REQUIRED:
                raise RuntimeError(
                    f"Insufficient global rows: "
                    f"{dataset_name}, seed={seed}, label={label}"
                )

            roles = (
                ["train"] * TRAIN_PER_CLASS
                + ["calibration"] * CALIBRATION_PER_CLASS
                + ["source_test"] * SOURCE_TEST_PER_CLASS
            )

            global_frames.append(
                pd.DataFrame({
                    "row_id": candidate_ids,
                    "Label": label,
                    "role": roles,
                })
            )

        global_index = pd.concat(
            global_frames,
            ignore_index=True,
        )

        # Integrity checks
        env_set = set(
            environment_index["row_id"].astype(int)
        )

        global_set = set(
            global_index["row_id"].astype(int)
        )

        if env_set.intersection(global_set):
            raise RuntimeError(
                f"Environment/global overlap for seed {seed}"
            )

        if global_index["row_id"].duplicated().any():
            raise RuntimeError(
                f"Duplicate global row IDs for seed {seed}"
            )

        expected_environment_rows = (
            number_of_blocks
            * 2
            * ENVIRONMENT_PER_CLASS
        )

        if len(environment_index) != expected_environment_rows:
            raise RuntimeError(
                "Unexpected environment sample size."
            )

        environment_index = environment_index.sort_values([
            "actual_environment_id",
            "Label",
            "row_id",
        ]).reset_index(drop=True)

        global_index = global_index.sort_values([
            "role",
            "Label",
            "row_id",
        ]).reset_index(drop=True)

        environment_file = (
            seed_directory / "environment_index.csv"
        )

        global_file = (
            seed_directory / "global_index.csv"
        )

        environment_index.to_csv(
            environment_file,
            index=False,
        )

        global_index.to_csv(
            global_file,
            index=False,
        )

        role_counts = (
            global_index.groupby(
                ["role", "Label"]
            ).size().to_dict()
        )

        seed_manifest = {
            "dataset": dataset_name,
            "seed": seed,
            "source_csv_rows": int(processed_rows),
            "eligible_environment_blocks": number_of_blocks,
            "environment_rows": int(
                len(environment_index)
            ),
            "train_rows": int(
                (global_index["role"] == "train").sum()
            ),
            "calibration_rows": int(
                (
                    global_index["role"]
                    == "calibration"
                ).sum()
            ),
            "source_test_rows": int(
                (
                    global_index["role"]
                    == "source_test"
                ).sum()
            ),
            "environment_global_overlap": 0,
            "environment_index_sha256": sha256_file(
                environment_file
            ),
            "global_index_sha256": sha256_file(
                global_file
            ),
            "protocol_sha256": EXPECTED_PROTOCOL_HASH,
            "role_label_counts": {
                f"{role}|{label}": int(count)
                for (role, label), count
                in role_counts.items()
            },
        }

        manifest_file = (
            seed_directory / "partition_manifest.json"
        )

        manifest_file.write_text(
            json.dumps(
                seed_manifest,
                indent=2,
                sort_keys=True,
            ),
            encoding="utf-8",
        )

        seed_manifests[str(seed)] = seed_manifest

        print(
            f"  Seed {seed}: "
            f"environment={len(environment_index):,}, "
            f"train=50,000, calibration=10,000, "
            f"source_test=10,000 — PASS"
        )

    completion = {
        "dataset": dataset_name,
        "protocol_sha256": EXPECTED_PROTOCOL_HASH,
        "source_rows": int(processed_rows),
        "eligible_blocks": number_of_blocks,
        "seeds": seed_manifests,
        "status": "PASS",
    }

    completion_file.write_text(
        json.dumps(
            completion,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    elapsed = (time.time() - start_time) / 60

    print(f"COMPLETED: {dataset_name}")
    print(f"Elapsed: {elapsed:.1f} minutes")
    print("Partition-index status: PASS")


print("\n" + "=" * 78)
print("ALL PAPER 3 PARTITION INDICES: PASS")
print(f"Output directory: {PARTITION_DIR}")
print("=" * 78)

PAPER 3 — DETERMINISTIC PARTITION-INDEX CONSTRUCTION
Expected protocol hash: e39a5007a7f2cbdc348d5179ce593af5766bc6688f5baa3a46a56399f683a488
Observed protocol hash: e39a5007a7f2cbdc348d5179ce593af5766bc6688f5baa3a46a56399f683a488
Protocol verification: PASS

------------------------------------------------------------------------------
DATASET: NF_BoT_IoT_v3
Eligible blocks: 5
Global candidate limit per class: 45,000
------------------------------------------------------------------------------
CSV member: d509c9db7490cf92_NFV3DATA-A11964_A11964/data/NF-BoT-IoT-v3.csv
  Chunks: 10 | Rows: 5,000,000 | Elapsed: 0.4 min
  Chunks: 20 | Rows: 10,000,000 | Elapsed: 0.7 min
  Chunks: 30 | Rows: 15,000,000 | Elapsed: 1.0 min
  Seed 2027: environment=20,000, train=50,000, calibration=10,000, source_test=10,000 — PASS
  Seed 2028: environment=20,000, train=50,000, calibration=10,000, source_test=10,000 — PASS
  Seed 2029: environment=20,000, train=50,000, calibration=10,000, source_test=10,000 

In [6]:
from pathlib import Path
import hashlib
import json
import time
import zipfile

import numpy as np
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

PROTOCOL_FILE = (
    ROOT / "Paper3_Protocol/paper3_protocol_frozen_v1.json"
)

PARTITION_DIR = ROOT / "Paper3_Partitions"
OUTPUT_DIR = ROOT / "Paper3_Materialized"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_PROTOCOL_HASH = (
    "e39a5007a7f2cbdc348d5179ce593af"
    "5766bc6688f5baa3a46a56399f683a488"
)

DATASETS = {
    "NF_BoT_IoT_v3": "NF-BoT-IoT-v3.zip",
    "NF_CSE_CIC_IDS2018_v3": "NF-CSE-CIC-IDS2018-v3.zip",
    "NF_ToN_IoT_v3": "NF-ToN-IoT-v3.zip",
    "NF_UNSW_NB15_v3": "NF-UNSW-NB15-v3.zip",
}

SEEDS = [2027, 2028, 2029, 2030, 2031]
CHUNK_SIZE = 250_000

EXCLUDED_COLUMNS = {
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "Label",
    "Attack",
}


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for part in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(part)

    return digest.hexdigest()


def atomic_parquet(frame, path):
    temporary = path.with_suffix(".temporary.parquet")

    frame.to_parquet(
        temporary,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    temporary.replace(path)


def feature_hashes(frame, feature_columns):
    return pd.util.hash_pandas_object(
        frame[feature_columns],
        index=False,
        categorize=True,
    ).to_numpy(dtype=np.uint64)


# ------------------------------------------------------------------
# 1. Verify frozen protocol
# ------------------------------------------------------------------

observed_protocol_hash = sha256_file(PROTOCOL_FILE)

if observed_protocol_hash != EXPECTED_PROTOCOL_HASH:
    raise RuntimeError(
        "Frozen protocol hash mismatch. Stop immediately."
    )

print("=" * 78)
print("PAPER 3 — FEATURE MATERIALIZATION AND OVERLAP AUDIT")
print("=" * 78)
print("Protocol verification: PASS")


# ------------------------------------------------------------------
# 2. Verify identical schema and freeze exact feature names
# ------------------------------------------------------------------

dataset_members = {}
dataset_features = {}

for dataset_name, zip_filename in DATASETS.items():
    zip_path = ROOT / zip_filename

    with zipfile.ZipFile(zip_path, "r") as archive:
        members = [
            member for member in archive.namelist()
            if member.lower().endswith(".csv")
            and "netflow_v3_features" not in member.lower()
            and not member.startswith("__MACOSX")
        ]

        csv_member = max(
            members,
            key=lambda member: archive.getinfo(member).file_size,
        )

        dataset_members[dataset_name] = csv_member

        with archive.open(csv_member) as csv_file:
            header = pd.read_csv(
                csv_file,
                nrows=0,
            ).columns.tolist()

        model_features = [
            column for column in header
            if column not in EXCLUDED_COLUMNS
        ]

        if len(model_features) != 49:
            raise RuntimeError(
                f"{dataset_name}: expected 49 predictors, "
                f"found {len(model_features)}."
            )

        dataset_features[dataset_name] = model_features

reference_features = dataset_features[
    "NF_BoT_IoT_v3"
]

for dataset_name, features in dataset_features.items():
    if features != reference_features:
        raise RuntimeError(
            f"Feature schema mismatch: {dataset_name}"
        )

feature_schema = {
    "schema_id": "P3_NF_V3_49_FEATURES_V1",
    "protocol_sha256": EXPECTED_PROTOCOL_HASH,
    "predictor_count": len(reference_features),
    "predictors_in_order": reference_features,
    "excluded_columns": sorted(EXCLUDED_COLUMNS),
}

schema_text = json.dumps(
    feature_schema,
    indent=2,
    sort_keys=True,
)

schema_file = OUTPUT_DIR / "feature_schema_v1.json"
schema_file.write_text(schema_text, encoding="utf-8")

schema_hash = sha256_file(schema_file)

(OUTPUT_DIR / "feature_schema_v1.sha256").write_text(
    schema_hash,
    encoding="utf-8",
)

print(f"Feature count: {len(reference_features)}")
print(f"Feature-schema SHA-256: {schema_hash}")
print("Cross-dataset feature-schema verification: PASS")


# ------------------------------------------------------------------
# 3. Materialize selected rows
# ------------------------------------------------------------------

for dataset_name, zip_filename in DATASETS.items():

    dataset_start = time.time()

    source_partition_dir = (
        PARTITION_DIR / dataset_name
    )

    dataset_output_dir = OUTPUT_DIR / dataset_name
    dataset_output_dir.mkdir(parents=True, exist_ok=True)

    completion_file = (
        dataset_output_dir / "materialization_complete.json"
    )

    if completion_file.exists():
        print(f"\nSKIPPED—already completed: {dataset_name}")
        continue

    print(f"\n{'-' * 78}")
    print(f"DATASET: {dataset_name}")
    print(f"{'-' * 78}")

    seed_indices = {}
    all_selected_ids = []

    for seed in SEEDS:
        seed_index_dir = (
            source_partition_dir / f"seed_{seed}"
        )

        environment_index = pd.read_csv(
            seed_index_dir / "environment_index.csv"
        )

        global_index = pd.read_csv(
            seed_index_dir / "global_index.csv"
        )

        environment_index["row_id"] = (
            environment_index["row_id"].astype("int64")
        )

        global_index["row_id"] = (
            global_index["row_id"].astype("int64")
        )

        seed_indices[seed] = {
            "environment": environment_index,
            "global": global_index,
        }

        all_selected_ids.append(
            environment_index["row_id"].to_numpy()
        )

        all_selected_ids.append(
            global_index["row_id"].to_numpy()
        )

    selected_union = np.unique(
        np.concatenate(all_selected_ids)
        .astype("int64")
    )

    print(
        f"Unique rows required across five seeds: "
        f"{len(selected_union):,}"
    )

    zip_path = ROOT / zip_filename
    csv_member = dataset_members[dataset_name]

    selected_parts = []
    processed_rows = 0

    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(csv_member) as csv_file:

            reader = pd.read_csv(
                csv_file,
                usecols=reference_features + ["Label"],
                chunksize=CHUNK_SIZE,
                low_memory=False,
            )

            for chunk_number, chunk in enumerate(
                reader,
                start=1,
            ):
                chunk_start = processed_rows
                chunk_end = processed_rows + len(chunk)

                left = np.searchsorted(
                    selected_union,
                    chunk_start,
                    side="left",
                )

                right = np.searchsorted(
                    selected_union,
                    chunk_end,
                    side="left",
                )

                required_ids = selected_union[left:right]

                if len(required_ids) > 0:
                    positions = (
                        required_ids - chunk_start
                    ).astype("int64")

                    selected_chunk = (
                        chunk.iloc[positions].copy()
                    )

                    selected_chunk.insert(
                        0,
                        "row_id",
                        required_ids,
                    )

                    selected_parts.append(selected_chunk)

                processed_rows = chunk_end

                if chunk_number % 10 == 0:
                    elapsed = (
                        time.time() - dataset_start
                    ) / 60

                    print(
                        f"  Chunks: {chunk_number:,} | "
                        f"Rows scanned: {processed_rows:,} | "
                        f"Elapsed: {elapsed:.1f} min"
                    )

    materialized_union = pd.concat(
        selected_parts,
        ignore_index=True,
    )

    materialized_union = (
        materialized_union
        .sort_values("row_id")
        .reset_index(drop=True)
    )

    if materialized_union["row_id"].duplicated().any():
        raise RuntimeError(
            f"{dataset_name}: duplicate materialized row IDs."
        )

    observed_ids = materialized_union[
        "row_id"
    ].to_numpy(dtype="int64")

    if not np.array_equal(
        observed_ids,
        selected_union,
    ):
        raise RuntimeError(
            f"{dataset_name}: selected row-ID mismatch."
        )

    # --------------------------------------------------------------
    # 4. Numeric conversion and quality audit
    # --------------------------------------------------------------

    feature_quality = {}
    total_infinite = 0
    total_coercion_missing = 0

    for feature in reference_features:
        original_missing = int(
            materialized_union[feature].isna().sum()
        )

        numeric_values = pd.to_numeric(
            materialized_union[feature],
            errors="coerce",
        )

        numeric_array = numeric_values.to_numpy(
            dtype=np.float64,
            na_value=np.nan,
        )

        infinite_mask = np.isinf(numeric_array)
        infinite_count = int(infinite_mask.sum())

        numeric_array[infinite_mask] = np.nan

        final_missing = int(
            np.isnan(numeric_array).sum()
        )

        coercion_missing = max(
            0,
            final_missing
            - original_missing
            - infinite_count,
        )

        materialized_union[feature] = (
            numeric_array.astype(np.float32)
        )

        feature_quality[feature] = {
            "original_missing": original_missing,
            "infinite_converted_to_missing": (
                infinite_count
            ),
            "non_numeric_converted_to_missing": (
                coercion_missing
            ),
            "final_missing": final_missing,
        }

        total_infinite += infinite_count
        total_coercion_missing += coercion_missing

    labels = pd.to_numeric(
        materialized_union["Label"],
        errors="coerce",
    )

    if labels.isna().any():
        raise RuntimeError(
            f"{dataset_name}: missing labels after conversion."
        )

    if not labels.isin([0, 1]).all():
        raise RuntimeError(
            f"{dataset_name}: invalid binary labels."
        )

    materialized_union["Label"] = (
        labels.astype(np.int8)
    )

    raw_label_lookup = (
        materialized_union
        .set_index("row_id")["Label"]
    )

    seed_manifests = {}

    # --------------------------------------------------------------
    # 5. Produce seed-specific Parquet files
    # --------------------------------------------------------------

    for seed in SEEDS:
        seed_output_dir = (
            dataset_output_dir / f"seed_{seed}"
        )
        seed_output_dir.mkdir(parents=True, exist_ok=True)

        environment_index = seed_indices[
            seed
        ]["environment"].copy()

        global_index = seed_indices[
            seed
        ]["global"].copy()

        expected_environment_labels = (
            environment_index["Label"]
            .astype(np.int8)
            .to_numpy()
        )

        observed_environment_labels = (
            raw_label_lookup.loc[
                environment_index["row_id"]
            ].to_numpy()
        )

        if not np.array_equal(
            expected_environment_labels,
            observed_environment_labels,
        ):
            raise RuntimeError(
                f"{dataset_name}, seed {seed}: "
                "environment label mismatch."
            )

        expected_global_labels = (
            global_index["Label"]
            .astype(np.int8)
            .to_numpy()
        )

        observed_global_labels = (
            raw_label_lookup.loc[
                global_index["row_id"]
            ].to_numpy()
        )

        if not np.array_equal(
            expected_global_labels,
            observed_global_labels,
        ):
            raise RuntimeError(
                f"{dataset_name}, seed {seed}: "
                "global label mismatch."
            )

        environment_data = (
            environment_index.drop(columns=["Label"])
            .merge(
                materialized_union,
                on="row_id",
                how="left",
                validate="one_to_one",
            )
        )

        global_data = (
            global_index.drop(columns=["Label"])
            .merge(
                materialized_union,
                on="row_id",
                how="left",
                validate="one_to_one",
            )
        )

        train_data = (
            global_data[
                global_data["role"] == "train"
            ]
            .drop(columns=["role"])
            .reset_index(drop=True)
        )

        calibration_data = (
            global_data[
                global_data["role"] == "calibration"
            ]
            .drop(columns=["role"])
            .reset_index(drop=True)
        )

        source_test_data = (
            global_data[
                global_data["role"] == "source_test"
            ]
            .drop(columns=["role"])
            .reset_index(drop=True)
        )

        environment_data = (
            environment_data.reset_index(drop=True)
        )

        expected_counts = {
            "train": 50_000,
            "calibration": 10_000,
            "source_test": 10_000,
        }

        observed_counts = {
            "train": len(train_data),
            "calibration": len(calibration_data),
            "source_test": len(source_test_data),
        }

        if observed_counts != expected_counts:
            raise RuntimeError(
                f"{dataset_name}, seed {seed}: "
                f"unexpected role counts {observed_counts}"
            )

        partition_frames = {
            "train": train_data,
            "calibration": calibration_data,
            "source_test": source_test_data,
            "environment": environment_data,
        }

        hash_sets = {}
        duplicate_rows = {}

        for partition_name, frame in partition_frames.items():
            hashes = feature_hashes(
                frame,
                reference_features,
            )

            hash_sets[partition_name] = set(
                hashes.tolist()
            )

            duplicate_rows[partition_name] = int(
                len(hashes) - len(np.unique(hashes))
            )

        partition_pairs = [
            ("train", "calibration"),
            ("train", "source_test"),
            ("train", "environment"),
            ("calibration", "source_test"),
            ("calibration", "environment"),
            ("source_test", "environment"),
        ]

        overlap_counts = {}

        for first, second in partition_pairs:
            key = f"{first}|{second}"

            overlap_counts[key] = len(
                hash_sets[first].intersection(
                    hash_sets[second]
                )
            )

        output_files = {}

        for partition_name, frame in partition_frames.items():
            output_file = (
                seed_output_dir
                / f"{partition_name}.parquet"
            )

            atomic_parquet(frame, output_file)

            output_files[partition_name] = {
                "path": str(output_file),
                "rows": int(len(frame)),
                "sha256": sha256_file(output_file),
            }

        seed_manifest = {
            "dataset": dataset_name,
            "seed": seed,
            "protocol_sha256": EXPECTED_PROTOCOL_HASH,
            "feature_schema_sha256": schema_hash,
            "output_files": output_files,
            "within_partition_duplicate_rows": (
                duplicate_rows
            ),
            "cross_partition_feature_hash_overlap": (
                overlap_counts
            ),
            "maximum_cross_partition_overlap": int(
                max(overlap_counts.values())
            ),
            "status": "PASS",
        }

        manifest_file = (
            seed_output_dir
            / "materialization_manifest.json"
        )

        manifest_file.write_text(
            json.dumps(
                seed_manifest,
                indent=2,
                sort_keys=True,
            ),
            encoding="utf-8",
        )

        seed_manifests[str(seed)] = seed_manifest

        print(
            f"  Seed {seed}: "
            f"train={len(train_data):,}, "
            f"cal={len(calibration_data):,}, "
            f"test={len(source_test_data):,}, "
            f"environment={len(environment_data):,}, "
            f"max-overlap="
            f"{max(overlap_counts.values()):,}"
        )

    quality_file = (
        dataset_output_dir
        / "feature_quality_audit.json"
    )

    quality_summary = {
        "dataset": dataset_name,
        "protocol_sha256": EXPECTED_PROTOCOL_HASH,
        "feature_schema_sha256": schema_hash,
        "scanned_source_rows": int(processed_rows),
        "unique_materialized_rows": int(
            len(materialized_union)
        ),
        "total_infinite_values_converted": int(
            total_infinite
        ),
        "total_non_numeric_values_converted": int(
            total_coercion_missing
        ),
        "per_feature": feature_quality,
    }

    quality_file.write_text(
        json.dumps(
            quality_summary,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    completion_summary = {
        "dataset": dataset_name,
        "protocol_sha256": EXPECTED_PROTOCOL_HASH,
        "feature_schema_sha256": schema_hash,
        "unique_materialized_rows": int(
            len(materialized_union)
        ),
        "quality_audit_sha256": sha256_file(
            quality_file
        ),
        "seeds": seed_manifests,
        "status": "PASS",
    }

    completion_file.write_text(
        json.dumps(
            completion_summary,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    elapsed = (time.time() - dataset_start) / 60

    print(f"COMPLETED: {dataset_name}")
    print(
        f"Unique materialized rows: "
        f"{len(materialized_union):,}"
    )
    print(
        f"Infinite values converted: "
        f"{total_infinite:,}"
    )
    print(
        f"Non-numeric values converted: "
        f"{total_coercion_missing:,}"
    )
    print(f"Elapsed: {elapsed:.1f} minutes")
    print("Materialization status: PASS")


print("\n" + "=" * 78)
print("ALL PAPER 3 FEATURE MATERIALIZATION: PASS")
print(f"Output directory: {OUTPUT_DIR}")
print("=" * 78)

PAPER 3 — FEATURE MATERIALIZATION AND OVERLAP AUDIT
Protocol verification: PASS
Feature count: 49
Feature-schema SHA-256: 67d61eebb3377e3053f943f1e4c031237b3ef35dd06296d8538fe552b16be0c6
Cross-dataset feature-schema verification: PASS

------------------------------------------------------------------------------
DATASET: NF_BoT_IoT_v3
------------------------------------------------------------------------------
Unique rows required across five seeds: 272,383
  Chunks: 10 | Rows scanned: 2,500,000 | Elapsed: 0.4 min
  Chunks: 20 | Rows scanned: 5,000,000 | Elapsed: 0.7 min
  Chunks: 30 | Rows scanned: 7,500,000 | Elapsed: 1.1 min
  Chunks: 40 | Rows scanned: 10,000,000 | Elapsed: 1.4 min
  Chunks: 50 | Rows scanned: 12,500,000 | Elapsed: 1.8 min
  Chunks: 60 | Rows scanned: 15,000,000 | Elapsed: 2.3 min
  Seed 2027: train=50,000, cal=10,000, test=10,000, environment=20,000, max-overlap=940
  Seed 2028: train=50,000, cal=10,000, test=10,000, environment=20,000, max-overlap=933
  Seed 2

In [7]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

PROTOCOL_DIR = ROOT / "Paper3_Protocol"
MATERIALIZED_DIR = ROOT / "Paper3_Materialized"
SANITIZED_DIR = ROOT / "Paper3_Sanitized"
SANITIZED_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL_FILE = (
    PROTOCOL_DIR / "paper3_protocol_frozen_v1.json"
)

SCHEMA_FILE = (
    MATERIALIZED_DIR / "feature_schema_v1.json"
)

EXPECTED_PROTOCOL_HASH = (
    "e39a5007a7f2cbdc348d5179ce593af"
    "5766bc6688f5baa3a46a56399f683a488"
)

EXPECTED_SCHEMA_HASH = (
    "67d61eebb3377e3053f943f1e4c03123"
    "7b3ef35dd06296d8538fe552b16be0c6"
)

DATASETS = [
    "NF_BoT_IoT_v3",
    "NF_CSE_CIC_IDS2018_v3",
    "NF_ToN_IoT_v3",
    "NF_UNSW_NB15_v3",
]

SEEDS = [2027, 2028, 2029, 2030, 2031]

ROLE_SALTS = {
    "source_test": np.uint64(0xA24BAED4963EE407),
    "calibration": np.uint64(0x9FB21C651E98DF25),
    "train": np.uint64(0xC13FA9A902A6328F),
}


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for part in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(part)

    return digest.hexdigest()


def atomic_parquet(frame, path):
    temporary = path.with_suffix(".temporary.parquet")

    frame.to_parquet(
        temporary,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    temporary.replace(path)


def feature_hashes(frame, feature_columns):
    return pd.util.hash_pandas_object(
        frame[feature_columns],
        index=False,
        categorize=True,
    ).to_numpy(dtype=np.uint64)


def splitmix64(values):
    values = values.astype(np.uint64, copy=False)

    with np.errstate(over="ignore"):
        values = values + np.uint64(
            0x9E3779B97F4A7C15
        )
        values = (
            values ^ np.right_shift(values, 30)
        ) * np.uint64(0xBF58476D1CE4E5B9)
        values = (
            values ^ np.right_shift(values, 27)
        ) * np.uint64(0x94D049BB133111EB)
        values = values ^ np.right_shift(values, 31)

    return values


def deterministic_balance(frame, seed, salt):
    class_counts = (
        frame["Label"].value_counts().to_dict()
    )

    if 0 not in class_counts or 1 not in class_counts:
        raise RuntimeError(
            "Both binary classes are required."
        )

    keep_per_class = min(
        int(class_counts[0]),
        int(class_counts[1]),
    )

    selected_parts = []

    for label in [0, 1]:
        class_frame = frame[
            frame["Label"] == label
        ].copy()

        row_ids = class_frame[
            "row_id"
        ].to_numpy(dtype=np.uint64)

        priorities = splitmix64(
            row_ids
            ^ np.uint64(seed)
            ^ salt
        )

        if len(class_frame) > keep_per_class:
            selected = np.argpartition(
                priorities,
                keep_per_class - 1,
            )[:keep_per_class]

            class_frame = class_frame.iloc[
                selected
            ].copy()

        selected_parts.append(class_frame)

    balanced = pd.concat(
        selected_parts,
        ignore_index=True,
    )

    return (
        balanced
        .sort_values(["Label", "row_id"])
        .reset_index(drop=True)
    )


def exclude_forbidden_hashes(
    frame,
    feature_columns,
    forbidden_hashes,
):
    if not forbidden_hashes:
        return frame.copy(), 0

    hashes = feature_hashes(
        frame,
        feature_columns,
    )

    forbidden_array = np.fromiter(
        forbidden_hashes,
        dtype=np.uint64,
    )

    keep_mask = ~np.isin(
        hashes,
        forbidden_array,
    )

    removed = int((~keep_mask).sum())

    return (
        frame.loc[keep_mask].reset_index(drop=True),
        removed,
    )


# ---------------------------------------------------------------
# 1. Verify frozen artifacts
# ---------------------------------------------------------------

if sha256_file(PROTOCOL_FILE) != EXPECTED_PROTOCOL_HASH:
    raise RuntimeError("Protocol hash mismatch.")

if sha256_file(SCHEMA_FILE) != EXPECTED_SCHEMA_HASH:
    raise RuntimeError("Feature-schema hash mismatch.")

with open(SCHEMA_FILE, "r", encoding="utf-8") as file:
    schema = json.load(file)

FEATURES = schema["predictors_in_order"]

if len(FEATURES) != 49:
    raise RuntimeError("Expected exactly 49 predictors.")

print("=" * 78)
print("PAPER 3 — DUPLICATE-GROUP SANITIZATION")
print("=" * 78)
print("Protocol verification: PASS")
print("Feature-schema verification: PASS")


# ---------------------------------------------------------------
# 2. Collect the complete observed overlap audit
# ---------------------------------------------------------------

overlap_rows = []

for dataset in DATASETS:
    for seed in SEEDS:
        manifest_file = (
            MATERIALIZED_DIR
            / dataset
            / f"seed_{seed}"
            / "materialization_manifest.json"
        )

        with open(
            manifest_file,
            "r",
            encoding="utf-8",
        ) as file:
            manifest = json.load(file)

        for pair, count in manifest[
            "cross_partition_feature_hash_overlap"
        ].items():
            overlap_rows.append({
                "dataset": dataset,
                "seed": seed,
                "partition_pair": pair,
                "overlapping_feature_hashes": int(count),
            })

overlap_table = pd.DataFrame(overlap_rows)

overlap_audit_file = (
    PROTOCOL_DIR
    / "pre_modelling_feature_overlap_audit.csv"
)

overlap_table.to_csv(
    overlap_audit_file,
    index=False,
)

observed_maximum = int(
    overlap_table[
        "overlapping_feature_hashes"
    ].max()
)

observed_total = int(
    overlap_table[
        "overlapping_feature_hashes"
    ].sum()
)

print(f"Maximum observed pair overlap: {observed_maximum:,}")
print(f"Sum across audited pairs: {observed_total:,}")


# ---------------------------------------------------------------
# 3. Create amendment before sanitization or modelling
# ---------------------------------------------------------------

amendment_file = (
    PROTOCOL_DIR
    / "amendment_A1_duplicate_group_disjoint.json"
)

amendment_hash_file = (
    PROTOCOL_DIR
    / "amendment_A1_duplicate_group_disjoint.sha256"
)

if amendment_file.exists():
    amendment_hash = sha256_file(amendment_file)

    if not amendment_hash_file.exists():
        raise RuntimeError(
            "Existing amendment has no hash file."
        )

    recorded_hash = (
        amendment_hash_file
        .read_text(encoding="utf-8")
        .strip()
    )

    if amendment_hash != recorded_hash:
        raise RuntimeError(
            "Existing amendment hash mismatch."
        )

else:
    amendment = {
        "amendment_id": (
            "P3_A1_DUPLICATE_GROUP_DISJOINT"
        ),
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "timing": (
            "Created after data-quality auditing but "
            "before model training, feature-importance "
            "estimation, or cross-dataset evaluation."
        ),
        "parent_protocol_sha256": (
            EXPECTED_PROTOCOL_HASH
        ),
        "feature_schema_sha256": (
            EXPECTED_SCHEMA_HASH
        ),
        "reason": (
            "The pre-modelling audit detected identical "
            "49-feature signatures across source partitions."
        ),
        "observed_maximum_pair_overlap": (
            observed_maximum
        ),
        "observed_total_across_pairs": (
            observed_total
        ),
        "primary_rule": {
            "group_definition": (
                "Identical cleaned values across all "
                "49 model predictors."
            ),
            "signature_method": (
                "Pandas deterministic 64-bit row hash; "
                "collision probability is negligible at "
                "the audited sample sizes."
            ),
            "partition_precedence": [
                "environment",
                "source_test",
                "calibration",
                "train",
            ],
            "action": (
                "When a feature signature occurs in more "
                "than one partition, retain it only in the "
                "highest-priority partition."
            ),
            "environment_samples": (
                "Preserved to retain the previously frozen "
                "2,000 observations per class per block."
            ),
            "within_partition_multiplicity": (
                "Retained in the primary analysis to preserve "
                "observed traffic frequency."
            ),
            "post_purge_balancing": (
                "Training, calibration and source-test "
                "partitions are independently downsampled "
                "to equal benign and attack counts."
            ),
            "replacement": False,
        },
        "unchanged_elements": [
            "research question",
            "four datasets",
            "six-hour environmental unit",
            "eligible blocks",
            "models",
            "five seeds",
            "49 predictors",
            "source-only warning scores",
            "target-blind evaluation",
            "primary outcome",
        ],
        "model_results_seen_before_amendment": False,
        "target_results_seen_before_amendment": False,
    }

    amendment_text = json.dumps(
        amendment,
        indent=2,
        sort_keys=True,
    )

    amendment_file.write_text(
        amendment_text,
        encoding="utf-8",
    )

    amendment_hash = sha256_file(amendment_file)

    amendment_hash_file.write_text(
        amendment_hash,
        encoding="utf-8",
    )

print(f"Amendment A1 SHA-256: {amendment_hash}")
print("Amendment timing verification: PRE-MODELLING")


# ---------------------------------------------------------------
# 4. Sanitize each dataset and seed
# ---------------------------------------------------------------

overall_rows = []

for dataset in DATASETS:

    dataset_output = SANITIZED_DIR / dataset
    dataset_output.mkdir(parents=True, exist_ok=True)

    completion_file = (
        dataset_output / "sanitization_complete.json"
    )

    if completion_file.exists():
        print(f"\nSKIPPED—already sanitized: {dataset}")
        continue

    print(f"\n{'-' * 78}")
    print(f"DATASET: {dataset}")
    print(f"{'-' * 78}")

    dataset_manifests = {}

    for seed in SEEDS:

        source_dir = (
            MATERIALIZED_DIR
            / dataset
            / f"seed_{seed}"
        )

        output_dir = (
            dataset_output / f"seed_{seed}"
        )
        output_dir.mkdir(parents=True, exist_ok=True)

        frames = {
            name: pd.read_parquet(
                source_dir / f"{name}.parquet"
            )
            for name in [
                "environment",
                "source_test",
                "calibration",
                "train",
            ]
        }

        original_counts = {
            name: int(len(frame))
            for name, frame in frames.items()
        }

        # Highest-priority partition remains unchanged.
        environment = frames["environment"].copy()

        environment_hashes = set(
            feature_hashes(
                environment,
                FEATURES,
            ).tolist()
        )

        # Source test loses environment-overlapping groups.
        source_test, test_overlap_removed = (
            exclude_forbidden_hashes(
                frames["source_test"],
                FEATURES,
                environment_hashes,
            )
        )

        before_balance = len(source_test)

        source_test = deterministic_balance(
            source_test,
            seed,
            ROLE_SALTS["source_test"],
        )

        test_balance_removed = (
            before_balance - len(source_test)
        )

        source_test_hashes = set(
            feature_hashes(
                source_test,
                FEATURES,
            ).tolist()
        )

        # Calibration loses environment and final test groups.
        calibration_forbidden = (
            environment_hashes
            | source_test_hashes
        )

        calibration, cal_overlap_removed = (
            exclude_forbidden_hashes(
                frames["calibration"],
                FEATURES,
                calibration_forbidden,
            )
        )

        before_balance = len(calibration)

        calibration = deterministic_balance(
            calibration,
            seed,
            ROLE_SALTS["calibration"],
        )

        cal_balance_removed = (
            before_balance - len(calibration)
        )

        calibration_hashes = set(
            feature_hashes(
                calibration,
                FEATURES,
            ).tolist()
        )

        # Training loses every evaluation/calibration group.
        train_forbidden = (
            environment_hashes
            | source_test_hashes
            | calibration_hashes
        )

        train, train_overlap_removed = (
            exclude_forbidden_hashes(
                frames["train"],
                FEATURES,
                train_forbidden,
            )
        )

        before_balance = len(train)

        train = deterministic_balance(
            train,
            seed,
            ROLE_SALTS["train"],
        )

        train_balance_removed = (
            before_balance - len(train)
        )

        sanitized_frames = {
            "environment": environment,
            "source_test": source_test,
            "calibration": calibration,
            "train": train,
        }

        final_hash_sets = {
            name: set(
                feature_hashes(
                    frame,
                    FEATURES,
                ).tolist()
            )
            for name, frame in sanitized_frames.items()
        }

        pairs = [
            ("train", "calibration"),
            ("train", "source_test"),
            ("train", "environment"),
            ("calibration", "source_test"),
            ("calibration", "environment"),
            ("source_test", "environment"),
        ]

        final_overlaps = {
            f"{first}|{second}": len(
                final_hash_sets[first]
                & final_hash_sets[second]
            )
            for first, second in pairs
        }

        if max(final_overlaps.values()) != 0:
            raise RuntimeError(
                f"{dataset}, seed {seed}: "
                "sanitization did not remove all overlap."
            )

        environment_counts = (
            environment.groupby([
                "actual_environment_id",
                "Label",
            ]).size()
        )

        if not (environment_counts == 2000).all():
            raise RuntimeError(
                f"{dataset}, seed {seed}: "
                "environment quota changed."
            )

        for role in [
            "train",
            "calibration",
            "source_test",
        ]:
            counts = (
                sanitized_frames[role]["Label"]
                .value_counts()
            )

            if counts.get(0, 0) != counts.get(1, 0):
                raise RuntimeError(
                    f"{dataset}, seed {seed}: "
                    f"{role} is not balanced."
                )

        output_files = {}

        for name, frame in sanitized_frames.items():
            output_file = output_dir / f"{name}.parquet"

            atomic_parquet(frame, output_file)

            output_files[name] = {
                "rows": int(len(frame)),
                "sha256": sha256_file(output_file),
                "path": str(output_file),
            }

        removal_counts = {
            "source_test_overlap_removed": int(
                test_overlap_removed
            ),
            "source_test_balance_removed": int(
                test_balance_removed
            ),
            "calibration_overlap_removed": int(
                cal_overlap_removed
            ),
            "calibration_balance_removed": int(
                cal_balance_removed
            ),
            "train_overlap_removed": int(
                train_overlap_removed
            ),
            "train_balance_removed": int(
                train_balance_removed
            ),
        }

        manifest = {
            "dataset": dataset,
            "seed": seed,
            "parent_protocol_sha256": (
                EXPECTED_PROTOCOL_HASH
            ),
            "amendment_A1_sha256": amendment_hash,
            "feature_schema_sha256": (
                EXPECTED_SCHEMA_HASH
            ),
            "original_rows": original_counts,
            "sanitized_rows": {
                name: int(len(frame))
                for name, frame
                in sanitized_frames.items()
            },
            "removal_counts": removal_counts,
            "final_cross_partition_overlap": (
                final_overlaps
            ),
            "maximum_final_overlap": 0,
            "environment_quota_preserved": True,
            "output_files": output_files,
            "status": "PASS",
        }

        manifest_file = (
            output_dir / "sanitization_manifest.json"
        )

        manifest_file.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            ),
            encoding="utf-8",
        )

        dataset_manifests[str(seed)] = manifest

        summary_row = {
            "dataset": dataset,
            "seed": seed,
            "train_rows": len(train),
            "calibration_rows": len(calibration),
            "source_test_rows": len(source_test),
            "environment_rows": len(environment),
            "overlap_removed": (
                test_overlap_removed
                + cal_overlap_removed
                + train_overlap_removed
            ),
            "final_overlap": 0,
        }

        overall_rows.append(summary_row)

        print(
            f"  Seed {seed}: "
            f"train={len(train):,}, "
            f"cal={len(calibration):,}, "
            f"test={len(source_test):,}, "
            f"environment={len(environment):,}, "
            f"overlap-removed="
            f"{summary_row['overlap_removed']:,}, "
            f"final-overlap=0 — PASS"
        )

    completion = {
        "dataset": dataset,
        "parent_protocol_sha256": (
            EXPECTED_PROTOCOL_HASH
        ),
        "amendment_A1_sha256": amendment_hash,
        "feature_schema_sha256": (
            EXPECTED_SCHEMA_HASH
        ),
        "seeds": dataset_manifests,
        "status": "PASS",
    }

    completion_file.write_text(
        json.dumps(
            completion,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    print(f"Sanitization completed: {dataset}")


summary_table = pd.DataFrame(overall_rows)

summary_file = (
    SANITIZED_DIR / "sanitization_summary.csv"
)

summary_table.to_csv(
    summary_file,
    index=False,
)

print("\n" + "=" * 78)
print("ALL PAPER 3 PARTITIONS FEATURE-GROUP-DISJOINT: PASS")
print(f"Amendment A1 SHA-256: {amendment_hash}")
print(f"Summary: {summary_file}")
print("=" * 78)

PAPER 3 — DUPLICATE-GROUP SANITIZATION
Protocol verification: PASS
Feature-schema verification: PASS
Maximum observed pair overlap: 1,563
Sum across audited pairs: 49,390
Amendment A1 SHA-256: 19c30a37d62d36f5b3c9bb795f64e10599dcef78a9c51bf96ca80b58d7db4b47
Amendment timing verification: PRE-MODELLING

------------------------------------------------------------------------------
DATASET: NF_BoT_IoT_v3
------------------------------------------------------------------------------
  Seed 2027: train=45,742, cal=9,260, test=9,546, environment=20,000, overlap-removed=2,734, final-overlap=0 — PASS
  Seed 2028: train=45,852, cal=9,288, test=9,560, environment=20,000, overlap-removed=2,655, final-overlap=0 — PASS
  Seed 2029: train=45,634, cal=9,312, test=9,542, environment=20,000, overlap-removed=2,762, final-overlap=0 — PASS
  Seed 2030: train=45,736, cal=9,302, test=9,538, environment=20,000, overlap-removed=2,714, final-overlap=0 — PASS
  Seed 2031: train=45,834, cal=9,264, test=9,556, e

In [8]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

PROTOCOL_DIR = ROOT / "Paper3_Protocol"
MATERIALIZED_DIR = ROOT / "Paper3_Materialized"
SANITIZED_DIR = ROOT / "Paper3_Sanitized"
AUDIT_DIR = ROOT / "Paper3_Pretraining_Audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_PROTOCOL_HASH = (
    "e39a5007a7f2cbdc348d5179ce593af"
    "5766bc6688f5baa3a46a56399f683a488"
)

EXPECTED_SCHEMA_HASH = (
    "67d61eebb3377e3053f943f1e4c03123"
    "7b3ef35dd06296d8538fe552b16be0c6"
)

EXPECTED_AMENDMENT_HASH = (
    "19c30a37d62d36f5b3c9bb795f64e105"
    "99dcef78a9c51bf96ca80b58d7db4b47"
)

DATASETS = [
    "NF_BoT_IoT_v3",
    "NF_CSE_CIC_IDS2018_v3",
    "NF_ToN_IoT_v3",
    "NF_UNSW_NB15_v3",
]

SEEDS = [2027, 2028, 2029, 2030, 2031]

PARTITIONS = [
    "train",
    "calibration",
    "source_test",
    "environment",
]


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for part in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(part)

    return digest.hexdigest()


# ---------------------------------------------------------------
# Verify protocol chain
# ---------------------------------------------------------------

protocol_file = (
    PROTOCOL_DIR / "paper3_protocol_frozen_v1.json"
)

schema_file = (
    MATERIALIZED_DIR / "feature_schema_v1.json"
)

amendment_file = (
    PROTOCOL_DIR
    / "amendment_A1_duplicate_group_disjoint.json"
)

if sha256_file(protocol_file) != EXPECTED_PROTOCOL_HASH:
    raise RuntimeError("Original protocol hash mismatch.")

if sha256_file(schema_file) != EXPECTED_SCHEMA_HASH:
    raise RuntimeError("Feature-schema hash mismatch.")

if sha256_file(amendment_file) != EXPECTED_AMENDMENT_HASH:
    raise RuntimeError("Amendment A1 hash mismatch.")

with open(schema_file, "r", encoding="utf-8") as file:
    FEATURES = json.load(file)["predictors_in_order"]

print("=" * 78)
print("PAPER 3 — FINAL PRE-TRAINING DATA GATE")
print("=" * 78)
print("Original protocol: PASS")
print("Feature schema: PASS")
print("Amendment A1: PASS")
print(f"Predictors: {len(FEATURES)}")


audit_rows = []
detailed_audit = {}
problems = []

for dataset in DATASETS:
    detailed_audit[dataset] = {}

    print(f"\n{'-' * 78}")
    print(f"DATASET: {dataset}")
    print(f"{'-' * 78}")

    for seed in SEEDS:
        seed_dir = (
            SANITIZED_DIR
            / dataset
            / f"seed_{seed}"
        )

        manifest_file = (
            seed_dir / "sanitization_manifest.json"
        )

        with open(
            manifest_file,
            "r",
            encoding="utf-8",
        ) as file:
            manifest = json.load(file)

        seed_details = {
            "partition_details": {},
            "problems": [],
        }

        hashes_valid = True
        total_infinite = 0

        for partition in PARTITIONS:
            parquet_file = (
                seed_dir / f"{partition}.parquet"
            )

            expected_hash = manifest[
                "output_files"
            ][partition]["sha256"]

            observed_hash = sha256_file(parquet_file)

            if observed_hash != expected_hash:
                hashes_valid = False
                seed_details["problems"].append(
                    f"{partition}_hash_mismatch"
                )

            frame = pd.read_parquet(parquet_file)

            missing_features = [
                feature for feature in FEATURES
                if feature not in frame.columns
            ]

            if missing_features:
                seed_details["problems"].append(
                    f"{partition}_missing_features"
                )
                problems.append(
                    f"{dataset} seed {seed} "
                    f"{partition}: missing predictors"
                )
                continue

            feature_array = frame[
                FEATURES
            ].to_numpy(dtype=np.float64)

            infinite_count = int(
                np.isinf(feature_array).sum()
            )

            total_infinite += infinite_count

            labels_valid = (
                frame["Label"].isin([0, 1]).all()
            )

            label_counts = (
                frame["Label"]
                .value_counts()
                .sort_index()
                .to_dict()
            )

            balanced = (
                int(label_counts.get(0, 0))
                == int(label_counts.get(1, 0))
            )

            missing_counts = (
                frame[FEATURES].isna().sum()
            )

            missing_fractions = (
                missing_counts / len(frame)
            )

            partition_details = {
                "rows": int(len(frame)),
                "label_0": int(
                    label_counts.get(0, 0)
                ),
                "label_1": int(
                    label_counts.get(1, 0)
                ),
                "balanced": bool(balanced),
                "labels_valid": bool(labels_valid),
                "infinite_values": infinite_count,
                "maximum_missing_fraction": float(
                    missing_fractions.max()
                ),
            }

            if partition == "train":
                all_missing_features = (
                    missing_counts[
                        missing_counts == len(frame)
                    ].index.tolist()
                )

                constant_features = []

                for feature in FEATURES:
                    if (
                        frame[feature]
                        .dropna()
                        .nunique()
                        <= 1
                    ):
                        constant_features.append(feature)

                partition_details[
                    "all_missing_features"
                ] = all_missing_features

                partition_details[
                    "constant_features"
                ] = constant_features

                if all_missing_features:
                    seed_details["problems"].append(
                        "all_missing_training_features"
                    )

            if infinite_count != 0:
                seed_details["problems"].append(
                    f"{partition}_contains_infinity"
                )

            if not labels_valid:
                seed_details["problems"].append(
                    f"{partition}_invalid_labels"
                )

            if not balanced:
                seed_details["problems"].append(
                    f"{partition}_not_balanced"
                )

            seed_details["partition_details"][
                partition
            ] = partition_details

        maximum_final_overlap = int(
            manifest["maximum_final_overlap"]
        )

        if maximum_final_overlap != 0:
            seed_details["problems"].append(
                "nonzero_cross_partition_overlap"
            )

        train_details = seed_details[
            "partition_details"
        ]["train"]

        gate_pass = (
            hashes_valid
            and total_infinite == 0
            and maximum_final_overlap == 0
            and not train_details[
                "all_missing_features"
            ]
            and not seed_details["problems"]
        )

        if not gate_pass:
            problems.append(
                f"{dataset} seed {seed}: "
                f"{seed_details['problems']}"
            )

        audit_rows.append({
            "dataset": dataset,
            "seed": seed,
            "train_rows": train_details["rows"],
            "calibration_rows": seed_details[
                "partition_details"
            ]["calibration"]["rows"],
            "source_test_rows": seed_details[
                "partition_details"
            ]["source_test"]["rows"],
            "environment_rows": seed_details[
                "partition_details"
            ]["environment"]["rows"],
            "constant_train_features": len(
                train_details["constant_features"]
            ),
            "all_missing_train_features": len(
                train_details[
                    "all_missing_features"
                ]
            ),
            "max_train_missing_pct": round(
                100
                * train_details[
                    "maximum_missing_fraction"
                ],
                4,
            ),
            "total_infinite": total_infinite,
            "file_hashes_valid": hashes_valid,
            "cross_partition_overlap": (
                maximum_final_overlap
            ),
            "gate_pass": gate_pass,
        })

        seed_details["gate_pass"] = gate_pass
        detailed_audit[dataset][str(seed)] = (
            seed_details
        )

        print(
            f"  Seed {seed}: "
            f"constant={len(train_details['constant_features'])}, "
            f"all-missing="
            f"{len(train_details['all_missing_features'])}, "
            f"max-missing="
            f"{100 * train_details['maximum_missing_fraction']:.4f}%, "
            f"infinite={total_infinite}, "
            f"overlap={maximum_final_overlap}, "
            f"hashes={'PASS' if hashes_valid else 'FAIL'}, "
            f"gate={'PASS' if gate_pass else 'FAIL'}"
        )


audit_table = pd.DataFrame(audit_rows)

audit_csv = AUDIT_DIR / "pretraining_gate_summary.csv"
audit_json = AUDIT_DIR / "pretraining_gate_details.json"

audit_table.to_csv(audit_csv, index=False)

audit_json.write_text(
    json.dumps(
        {
            "protocol_sha256": EXPECTED_PROTOCOL_HASH,
            "schema_sha256": EXPECTED_SCHEMA_HASH,
            "amendment_A1_sha256": (
                EXPECTED_AMENDMENT_HASH
            ),
            "details": detailed_audit,
            "problems": problems,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

overall_pass = (
    audit_table["gate_pass"].all()
    and len(problems) == 0
)

print("\n" + "=" * 78)
print(
    "FINAL PRE-TRAINING GATE: "
    f"{'PASS' if overall_pass else 'FAIL'}"
)
print(f"Problems: {problems}")
print(f"Summary: {audit_csv}")
print("=" * 78)

if not overall_pass:
    raise RuntimeError(
        "Pre-training gate failed. Do not train models."
    )

PAPER 3 — FINAL PRE-TRAINING DATA GATE
Original protocol: PASS
Feature schema: PASS
Amendment A1: PASS
Predictors: 49

------------------------------------------------------------------------------
DATASET: NF_BoT_IoT_v3
------------------------------------------------------------------------------
  Seed 2027: constant=0, all-missing=0, max-missing=33.3370%, infinite=0, overlap=0, hashes=PASS, gate=PASS
  Seed 2028: constant=0, all-missing=0, max-missing=33.2766%, infinite=0, overlap=0, hashes=PASS, gate=PASS
  Seed 2029: constant=0, all-missing=0, max-missing=33.2756%, infinite=0, overlap=0, hashes=PASS, gate=PASS
  Seed 2030: constant=0, all-missing=0, max-missing=33.2932%, infinite=0, overlap=0, hashes=PASS, gate=PASS
  Seed 2031: constant=0, all-missing=0, max-missing=33.3682%, infinite=0, overlap=0, hashes=PASS, gate=PASS

------------------------------------------------------------------------------
DATASET: NF_CSE_CIC_IDS2018_v3
-------------------------------------------------

In [10]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    recall_score,
    roc_auc_score,
    roc_curve,
)

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

PROTOCOL_DIR = ROOT / "Paper3_Protocol"
MATERIALIZED_DIR = ROOT / "Paper3_Materialized"
SANITIZED_DIR = ROOT / "Paper3_Sanitized"
MODEL_DIR = ROOT / "Paper3_Source_Models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_PROTOCOL_HASH = (
    "e39a5007a7f2cbdc348d5179ce593af"
    "5766bc6688f5baa3a46a56399f683a488"
)

EXPECTED_SCHEMA_HASH = (
    "67d61eebb3377e3053f943f1e4c03123"
    "7b3ef35dd06296d8538fe552b16be0c6"
)

EXPECTED_AMENDMENT_HASH = (
    "19c30a37d62d36f5b3c9bb795f64e105"
    "99dcef78a9c51bf96ca80b58d7db4b47"
)

DATASETS = [
    "NF_BoT_IoT_v3",
    "NF_CSE_CIC_IDS2018_v3",
    "NF_ToN_IoT_v3",
    "NF_UNSW_NB15_v3",
]

SEEDS = [2027, 2028, 2029, 2030, 2031]


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for part in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(part)

    return digest.hexdigest()


def atomic_joblib(obj, path):
    temporary = path.with_suffix(".temporary.joblib")
    joblib.dump(obj, temporary, compress=3)
    temporary.replace(path)


def atomic_parquet(frame, path):
    temporary = path.with_suffix(".temporary.parquet")

    frame.to_parquet(
        temporary,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    temporary.replace(path)


def select_balanced_accuracy_threshold(y_true, probabilities):
    fpr, tpr, thresholds = roc_curve(
        y_true,
        probabilities,
        drop_intermediate=False,
    )

    valid = (
        np.isfinite(thresholds)
        & (thresholds >= 0)
        & (thresholds <= 1)
    )

    fpr = fpr[valid]
    tpr = tpr[valid]
    thresholds = thresholds[valid]

    balanced_accuracies = (
        tpr + (1.0 - fpr)
    ) / 2.0

    best_value = float(
        np.max(balanced_accuracies)
    )

    best_indices = np.flatnonzero(
        np.isclose(
            balanced_accuracies,
            best_value,
            rtol=0,
            atol=1e-12,
        )
    )

    # Frozen tie-break: threshold closest to 0.5
    distances = np.abs(
        thresholds[best_indices] - 0.5
    )

    chosen_local = best_indices[
        np.argmin(distances)
    ]

    return {
        "threshold": float(
            thresholds[chosen_local]
        ),
        "calibration_balanced_accuracy": (
            best_value
        ),
        "number_of_tied_optima": int(
            len(best_indices)
        ),
    }


def calculate_metrics(y_true, probabilities, threshold):
    predictions = (
        probabilities >= threshold
    ).astype(np.int8)

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(y_true, predictions)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),
        "pr_auc_attack": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "benign_recall": float(
            recall_score(
                y_true,
                predictions,
                pos_label=0,
                zero_division=0,
            )
        ),
        "attack_recall": float(
            recall_score(
                y_true,
                predictions,
                pos_label=1,
                zero_division=0,
            )
        ),
    }


def predict_probability(model, features):
    probabilities = model.predict_proba(features)
    return probabilities[:, 1]


# ----------------------------------------------------------------
# Verify protocol chain
# ----------------------------------------------------------------

protocol_file = (
    PROTOCOL_DIR / "paper3_protocol_frozen_v1.json"
)

schema_file = (
    MATERIALIZED_DIR / "feature_schema_v1.json"
)

amendment_file = (
    PROTOCOL_DIR
    / "amendment_A1_duplicate_group_disjoint.json"
)

if sha256_file(protocol_file) != EXPECTED_PROTOCOL_HASH:
    raise RuntimeError("Protocol hash mismatch.")

if sha256_file(schema_file) != EXPECTED_SCHEMA_HASH:
    raise RuntimeError("Feature-schema hash mismatch.")

if sha256_file(amendment_file) != EXPECTED_AMENDMENT_HASH:
    raise RuntimeError("Amendment A1 hash mismatch.")

with open(schema_file, "r", encoding="utf-8") as file:
    FEATURES = json.load(file)["predictors_in_order"]

software_versions = {
    "python_note": "Google Colab runtime",
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "joblib": joblib.__version__,
}

print("=" * 78)
print("PAPER 3 — SOURCE-ONLY MODEL TRAINING")
print("=" * 78)
print("Protocol chain verification: PASS")
print(f"Predictors: {len(FEATURES)}")
print(f"Software versions: {software_versions}")


all_result_rows = []
all_training_problems = []


for dataset in DATASETS:

    print(f"\n{'-' * 78}")
    print(f"DATASET: {dataset}")
    print(f"{'-' * 78}")

    for seed in SEEDS:

        seed_start = time.time()

        input_dir = (
            SANITIZED_DIR
            / dataset
            / f"seed_{seed}"
        )

        output_dir = (
            MODEL_DIR
            / dataset
            / f"seed_{seed}"
        )
        output_dir.mkdir(parents=True, exist_ok=True)

        completion_file = (
            output_dir / "source_seed_complete.json"
        )

        if completion_file.exists():
            with open(
                completion_file,
                "r",
                encoding="utf-8",
            ) as file:
                completed = json.load(file)

            for model_name, model_result in completed[
                "models"
            ].items():
                all_result_rows.append({
                    "dataset": dataset,
                    "seed": seed,
                    "model": model_name,
                    **model_result["source_test_metrics"],
                })

            print(f"  Seed {seed}: SKIPPED—already completed")
            continue

        sanitized_manifest_file = (
            input_dir / "sanitization_manifest.json"
        )

        with open(
            sanitized_manifest_file,
            "r",
            encoding="utf-8",
        ) as file:
            sanitized_manifest = json.load(file)

        if sanitized_manifest[
            "maximum_final_overlap"
        ] != 0:
            raise RuntimeError(
                f"{dataset}, seed {seed}: overlap is not zero."
            )

        required_files = {
            name: input_dir / f"{name}.parquet"
            for name in [
                "train",
                "calibration",
                "source_test",
            ]
        }

        for name, path in required_files.items():
            expected_hash = sanitized_manifest[
                "output_files"
            ][name]["sha256"]

            if sha256_file(path) != expected_hash:
                raise RuntimeError(
                    f"{dataset}, seed {seed}: "
                    f"{name} hash mismatch."
                )

        train = pd.read_parquet(
            required_files["train"]
        )

        calibration = pd.read_parquet(
            required_files["calibration"]
        )

        source_test = pd.read_parquet(
            required_files["source_test"]
        )

        X_train_raw = train[
            FEATURES
        ].to_numpy(dtype=np.float32)

        y_train = train[
            "Label"
        ].to_numpy(dtype=np.int8)

        X_calibration_raw = calibration[
            FEATURES
        ].to_numpy(dtype=np.float32)

        y_calibration = calibration[
            "Label"
        ].to_numpy(dtype=np.int8)

        X_source_test_raw = source_test[
            FEATURES
        ].to_numpy(dtype=np.float32)

        y_source_test = source_test[
            "Label"
        ].to_numpy(dtype=np.int8)

        # Fit only on source training data
        imputer = SimpleImputer(
            strategy="median",
        )

        X_train = imputer.fit_transform(
            X_train_raw
        ).astype(np.float32)

        X_calibration = imputer.transform(
            X_calibration_raw
        ).astype(np.float32)

        X_source_test = imputer.transform(
            X_source_test_raw
        ).astype(np.float32)

        if (
            np.isnan(X_train).any()
            or np.isnan(X_calibration).any()
            or np.isnan(X_source_test).any()
        ):
            raise RuntimeError(
                f"{dataset}, seed {seed}: "
                "missing values remain after imputation."
            )

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(
            X_train
        ).astype(np.float32)

        X_calibration_scaled = scaler.transform(
            X_calibration
        ).astype(np.float32)

        X_source_test_scaled = scaler.transform(
            X_source_test
        ).astype(np.float32)

        preprocessor_artifact = {
            "dataset": dataset,
            "seed": seed,
            "features": FEATURES,
            "imputer": imputer,
            "logistic_regression_scaler": scaler,
            "protocol_sha256": EXPECTED_PROTOCOL_HASH,
            "amendment_A1_sha256": (
                EXPECTED_AMENDMENT_HASH
            ),
            "feature_schema_sha256": (
                EXPECTED_SCHEMA_HASH
            ),
        }

        preprocessor_file = (
            output_dir / "source_preprocessor.joblib"
        )

        atomic_joblib(
            preprocessor_artifact,
            preprocessor_file,
        )

        model_definitions = {
            "LogisticRegression": {
                "model": LogisticRegression(
                    solver="liblinear",
                    max_iter=2000,
                    class_weight=None,
                    random_state=seed,
                ),
                "scaled": True,
            },

            "RandomForest": {
                "model": RandomForestClassifier(
                    n_estimators=300,
                    max_features="sqrt",
                    min_samples_leaf=2,
                    class_weight=None,
                    random_state=seed,
                    n_jobs=-1,
                ),
                "scaled": False,
            },

            "XGBoost": {
                "model": XGBClassifier(
                    n_estimators=600,
                    learning_rate=0.05,
                    max_depth=6,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    tree_method="hist",
                    random_state=seed,
                    n_jobs=-1,
                    verbosity=0,
                ),
                "scaled": False,
            },
        }

        seed_model_results = {}

        for model_name, definition in model_definitions.items():

            model_start = time.time()
            model = definition["model"]
            uses_scaling = definition["scaled"]

            if uses_scaling:
                fit_features = X_train_scaled
                calibration_features = (
                    X_calibration_scaled
                )
                source_test_features = (
                    X_source_test_scaled
                )
            else:
                fit_features = X_train
                calibration_features = X_calibration
                source_test_features = X_source_test

            with warnings.catch_warnings(
                record=True
            ) as captured_warnings:
                warnings.simplefilter("always")

                model.fit(
                    fit_features,
                    y_train,
                )

            warning_messages = [
                str(item.message)
                for item in captured_warnings
            ]

            convergence_warnings = [
                item for item in captured_warnings
                if issubclass(
                    item.category,
                    ConvergenceWarning,
                )
            ]

            calibration_probabilities = (
                predict_probability(
                    model,
                    calibration_features,
                )
            )

            threshold_result = (
                select_balanced_accuracy_threshold(
                    y_calibration,
                    calibration_probabilities,
                )
            )

            threshold = threshold_result[
                "threshold"
            ]

            source_test_probabilities = (
                predict_probability(
                    model,
                    source_test_features,
                )
            )

            calibration_metrics = calculate_metrics(
                y_calibration,
                calibration_probabilities,
                threshold,
            )

            source_test_metrics = calculate_metrics(
                y_source_test,
                source_test_probabilities,
                threshold,
            )

            calibration_predictions = pd.DataFrame({
                "row_id": calibration[
                    "row_id"
                ].to_numpy(dtype=np.int64),
                "y_true": y_calibration,
                "probability_attack": (
                    calibration_probabilities
                ),
                "y_pred": (
                    calibration_probabilities
                    >= threshold
                ).astype(np.int8),
            })

            source_test_predictions = pd.DataFrame({
                "row_id": source_test[
                    "row_id"
                ].to_numpy(dtype=np.int64),
                "y_true": y_source_test,
                "probability_attack": (
                    source_test_probabilities
                ),
                "y_pred": (
                    source_test_probabilities
                    >= threshold
                ).astype(np.int8),
            })

            model_file = (
                output_dir
                / f"{model_name}_model.joblib"
            )

            calibration_prediction_file = (
                output_dir
                / f"{model_name}_calibration_predictions.parquet"
            )

            source_test_prediction_file = (
                output_dir
                / f"{model_name}_source_test_predictions.parquet"
            )

            model_artifact = {
                "dataset": dataset,
                "seed": seed,
                "model_name": model_name,
                "model": model,
                "uses_standard_scaler": uses_scaling,
                "threshold": threshold,
                "features": FEATURES,
                "protocol_sha256": (
                    EXPECTED_PROTOCOL_HASH
                ),
                "amendment_A1_sha256": (
                    EXPECTED_AMENDMENT_HASH
                ),
                "feature_schema_sha256": (
                    EXPECTED_SCHEMA_HASH
                ),
            }

            atomic_joblib(
                model_artifact,
                model_file,
            )

            atomic_parquet(
                calibration_predictions,
                calibration_prediction_file,
            )

            atomic_parquet(
                source_test_predictions,
                source_test_prediction_file,
            )

            model_elapsed = (
                time.time() - model_start
            )

            model_gate_pass = (
                len(convergence_warnings) == 0
                and np.isfinite(
                    calibration_probabilities
                ).all()
                and np.isfinite(
                    source_test_probabilities
                ).all()
                and 0 <= threshold <= 1
            )

            if not model_gate_pass:
                all_training_problems.append(
                    f"{dataset}, seed {seed}, "
                    f"{model_name}: model gate failed"
                )

            model_result = {
                "model_name": model_name,
                "uses_standard_scaler": (
                    uses_scaling
                ),
                "threshold_selection": (
                    threshold_result
                ),
                "calibration_metrics": (
                    calibration_metrics
                ),
                "source_test_metrics": (
                    source_test_metrics
                ),
                "fit_warning_count": int(
                    len(warning_messages)
                ),
                "fit_warnings": warning_messages,
                "convergence_warning_count": int(
                    len(convergence_warnings)
                ),
                "elapsed_seconds": round(
                    model_elapsed,
                    3,
                ),
                "model_gate_pass": (
                    model_gate_pass
                ),
                "files": {
                    "model": {
                        "path": str(model_file),
                        "sha256": sha256_file(
                            model_file
                        ),
                    },
                    "calibration_predictions": {
                        "path": str(
                            calibration_prediction_file
                        ),
                        "sha256": sha256_file(
                            calibration_prediction_file
                        ),
                    },
                    "source_test_predictions": {
                        "path": str(
                            source_test_prediction_file
                        ),
                        "sha256": sha256_file(
                            source_test_prediction_file
                        ),
                    },
                },
            }

            seed_model_results[
                model_name
            ] = model_result

            all_result_rows.append({
                "dataset": dataset,
                "seed": seed,
                "model": model_name,
                **source_test_metrics,
            })

            print(
                f"  Seed {seed} | {model_name}: "
                f"threshold={threshold:.6f}, "
                f"source BA="
                f"{source_test_metrics['balanced_accuracy']:.4f}, "
                f"Macro F1="
                f"{source_test_metrics['macro_f1']:.4f}, "
                f"MCC={source_test_metrics['mcc']:.4f}, "
                f"warnings={len(warning_messages)}, "
                f"gate="
                f"{'PASS' if model_gate_pass else 'FAIL'}"
            )

        seed_completion = {
            "dataset": dataset,
            "seed": seed,
            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "protocol_sha256": (
                EXPECTED_PROTOCOL_HASH
            ),
            "amendment_A1_sha256": (
                EXPECTED_AMENDMENT_HASH
            ),
            "feature_schema_sha256": (
                EXPECTED_SCHEMA_HASH
            ),
            "input_sanitization_manifest_sha256": (
                sha256_file(
                    sanitized_manifest_file
                )
            ),
            "preprocessor": {
                "path": str(preprocessor_file),
                "sha256": sha256_file(
                    preprocessor_file
                ),
            },
            "software_versions": software_versions,
            "models": seed_model_results,
            "elapsed_seconds": round(
                time.time() - seed_start,
                3,
            ),
            "status": (
                "PASS"
                if all(
                    item["model_gate_pass"]
                    for item in seed_model_results.values()
                )
                else "FAIL"
            ),
        }

        completion_file.write_text(
            json.dumps(
                seed_completion,
                indent=2,
                sort_keys=True,
            ),
            encoding="utf-8",
        )

        print(
            f"  Seed {seed} completed in "
            f"{(time.time() - seed_start) / 60:.1f} min"
        )


# ----------------------------------------------------------------
# Source-results summary and cryptographic seal
# ----------------------------------------------------------------

results_table = pd.DataFrame(all_result_rows)

results_file = (
    MODEL_DIR / "source_test_metrics_all_models.csv"
)

results_table.to_csv(
    results_file,
    index=False,
)

seal_entries = []

for dataset in DATASETS:
    for seed in SEEDS:
        completion_file = (
            MODEL_DIR
            / dataset
            / f"seed_{seed}"
            / "source_seed_complete.json"
        )

        if not completion_file.exists():
            raise RuntimeError(
                f"Missing completion file: "
                f"{dataset}, seed {seed}"
            )

        with open(
            completion_file,
            "r",
            encoding="utf-8",
        ) as file:
            completion = json.load(file)

        if completion["status"] != "PASS":
            all_training_problems.append(
                f"{dataset}, seed {seed}: "
                "completion status failed"
            )

        seal_entries.append({
            "relative_path": str(
                completion_file.relative_to(
                    MODEL_DIR
                )
            ),
            "sha256": sha256_file(
                completion_file
            ),
        })

        preprocessor_path = Path(
            completion["preprocessor"]["path"]
        )

        seal_entries.append({
            "relative_path": str(
                preprocessor_path.relative_to(
                    MODEL_DIR
                )
            ),
            "sha256": sha256_file(
                preprocessor_path
            ),
        })

        for model_result in completion[
            "models"
        ].values():
            for artifact in model_result[
                "files"
            ].values():
                artifact_path = Path(
                    artifact["path"]
                )

                seal_entries.append({
                    "relative_path": str(
                        artifact_path.relative_to(
                            MODEL_DIR
                        )
                    ),
                    "sha256": sha256_file(
                        artifact_path
                    ),
                })

seal_entries.append({
    "relative_path": results_file.name,
    "sha256": sha256_file(results_file),
})

seal_manifest = {
    "seal_id": "P3_SOURCE_MODELS_SEAL_V1",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "protocol_sha256": EXPECTED_PROTOCOL_HASH,
    "amendment_A1_sha256": EXPECTED_AMENDMENT_HASH,
    "feature_schema_sha256": EXPECTED_SCHEMA_HASH,
    "source_models": 60,
    "target_datasets_evaluated": 0,
    "target_results_seen": False,
    "artifacts": sorted(
        seal_entries,
        key=lambda item: item["relative_path"],
    ),
    "training_problems": all_training_problems,
    "status": (
        "PASS"
        if not all_training_problems
        else "FAIL"
    ),
}

seal_file = (
    MODEL_DIR / "source_model_seal_manifest.json"
)

seal_file.write_text(
    json.dumps(
        seal_manifest,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

seal_hash = sha256_file(seal_file)

(
    MODEL_DIR
    / "source_model_seal_manifest.sha256"
).write_text(
    seal_hash,
    encoding="utf-8",
)

print("\n" + "=" * 78)
print(
    "SOURCE-ONLY MODEL TRAINING AND SEAL: "
    f"{seal_manifest['status']}"
)
print(f"Models trained: {len(results_table)}")
print(f"Problems: {all_training_problems}")
print(f"Source-model seal SHA-256: {seal_hash}")
print(f"Source metrics: {results_file}")
print("=" * 78)

if seal_manifest["status"] != "PASS":
    raise RuntimeError(
        "Source-model seal failed. Do not continue."
    )

PAPER 3 — SOURCE-ONLY MODEL TRAINING
Protocol chain verification: PASS
Predictors: 49
Software versions: {'python_note': 'Google Colab runtime', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit_learn': '1.6.1', 'xgboost': '3.4.1', 'joblib': '1.5.3'}

------------------------------------------------------------------------------
DATASET: NF_BoT_IoT_v3
------------------------------------------------------------------------------
  Seed 2027: SKIPPED—already completed
  Seed 2028: SKIPPED—already completed
  Seed 2029: SKIPPED—already completed
  Seed 2030: SKIPPED—already completed
  Seed 2031: SKIPPED—already completed

------------------------------------------------------------------------------
DATASET: NF_CSE_CIC_IDS2018_v3
------------------------------------------------------------------------------
  Seed 2027: SKIPPED—already completed
  Seed 2028: SKIPPED—already completed
  Seed 2029: SKIPPED—already completed
  Seed 2030: SKIPPED—already completed
  Seed 2031: SKIPPED—already co

In [11]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import hashlib
import json
import time

import joblib
import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score

ROOT = Path(
    "/content/drivesyz/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

# Correct the path if the accidental "drivesyz" text is present.
ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

PROTOCOL_DIR = ROOT / "Paper3_Protocol"
MATERIALIZED_DIR = ROOT / "Paper3_Materialized"
SANITIZED_DIR = ROOT / "Paper3_Sanitized"
MODEL_DIR = ROOT / "Paper3_Source_Models"
STABILITY_DIR = ROOT / "Paper3_Source_Stability"
STABILITY_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_PROTOCOL_HASH = (
    "e39a5007a7f2cbdc348d5179ce593af"
    "5766bc6688f5baa3a46a56399f683a488"
)

EXPECTED_SCHEMA_HASH = (
    "67d61eebb3377e3053f943f1e4c03123"
    "7b3ef35dd06296d8538fe552b16be0c6"
)

EXPECTED_AMENDMENT_HASH = (
    "19c30a37d62d36f5b3c9bb795f64e105"
    "99dcef78a9c51bf96ca80b58d7db4b47"
)

EXPECTED_SOURCE_MODEL_SEAL = (
    "81eebf7b013fec0ca1802ebb54894fe73"
    "d7023b2c6450da09c72af3b3f5a3182"
)

DATASETS = [
    "NF_BoT_IoT_v3",
    "NF_CSE_CIC_IDS2018_v3",
    "NF_ToN_IoT_v3",
    "NF_UNSW_NB15_v3",
]

SEEDS = [2027, 2028, 2029, 2030, 2031]

MODELS = [
    "LogisticRegression",
    "XGBoost",
    "RandomForest",
]

N_REPEATS = 3
TOP_K = 10


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for part in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(part)

    return digest.hexdigest()


def atomic_parquet(frame, path):
    temporary = path.with_suffix(".temporary.parquet")

    frame.to_parquet(
        temporary,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    temporary.replace(path)


def stable_random_state(*parts):
    text = "|".join(str(part) for part in parts)

    digest = hashlib.sha256(
        text.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )


def threshold_scorer(threshold):
    def scorer(estimator, X, y):
        probabilities = estimator.predict_proba(X)[:, 1]

        predictions = (
            probabilities >= threshold
        ).astype(np.int8)

        return balanced_accuracy_score(
            y,
            predictions,
        )

    return scorer


def calculate_pairwise_stability(
    importance_frame,
    feature_names,
    group_type,
):
    subset = importance_frame[
        importance_frame["group_type"] == group_type
    ].copy()

    pivot = subset.pivot(
        index="group_id",
        columns="feature",
        values="importance_mean",
    )

    pivot = pivot.reindex(
        columns=feature_names
    )

    if pivot.isna().any().any():
        raise RuntimeError(
            f"Missing importance values for {group_type}"
        )

    group_ids = pivot.index.tolist()
    pairwise_rows = []

    for first, second in combinations(group_ids, 2):
        first_values = pivot.loc[
            first
        ].to_numpy(dtype=np.float64)

        second_values = pivot.loc[
            second
        ].to_numpy(dtype=np.float64)

        # Average ranks preserve ties.
        first_ranks = (
            pd.Series(-first_values)
            .rank(method="average")
            .to_numpy(dtype=np.float64)
        )

        second_ranks = (
            pd.Series(-second_values)
            .rank(method="average")
            .to_numpy(dtype=np.float64)
        )

        first_std = float(
            np.std(first_ranks)
        )

        second_std = float(
            np.std(second_ranks)
        )

        undefined_spearman = (
            first_std == 0
            or second_std == 0
        )

        if undefined_spearman:
            # No ranking information is treated as zero stability.
            spearman = 0.0
        else:
            spearman = float(
                np.corrcoef(
                    first_ranks,
                    second_ranks,
                )[0, 1]
            )

        first_order = sorted(
            range(len(feature_names)),
            key=lambda index: (
                -first_values[index],
                feature_names[index],
            ),
        )[:TOP_K]

        second_order = sorted(
            range(len(feature_names)),
            key=lambda index: (
                -second_values[index],
                feature_names[index],
            ),
        )[:TOP_K]

        first_top = {
            feature_names[index]
            for index in first_order
        }

        second_top = {
            feature_names[index]
            for index in second_order
        }

        top_k_jaccard = (
            len(first_top & second_top)
            / len(first_top | second_top)
        )

        pairwise_rows.append({
            "group_type": group_type,
            "group_1": str(first),
            "group_2": str(second),
            "spearman": spearman,
            "top10_jaccard": float(
                top_k_jaccard
            ),
            "undefined_spearman": bool(
                undefined_spearman
            ),
        })

    pairwise_frame = pd.DataFrame(
        pairwise_rows
    )

    if pairwise_frame.empty:
        raise RuntimeError(
            f"Fewer than two groups for {group_type}"
        )

    summary = {
        "groups": int(len(group_ids)),
        "pairs": int(len(pairwise_frame)),
        "mean_spearman": float(
            pairwise_frame["spearman"].mean()
        ),
        "median_spearman": float(
            pairwise_frame["spearman"].median()
        ),
        "mean_top10_jaccard": float(
            pairwise_frame[
                "top10_jaccard"
            ].mean()
        ),
        "undefined_spearman_pairs": int(
            pairwise_frame[
                "undefined_spearman"
            ].sum()
        ),
    }

    return pairwise_frame, summary


# ----------------------------------------------------------------
# Verify the entire source-only artifact chain
# ----------------------------------------------------------------

protocol_file = (
    PROTOCOL_DIR / "paper3_protocol_frozen_v1.json"
)

schema_file = (
    MATERIALIZED_DIR / "feature_schema_v1.json"
)

amendment_file = (
    PROTOCOL_DIR
    / "amendment_A1_duplicate_group_disjoint.json"
)

source_model_seal_file = (
    MODEL_DIR / "source_model_seal_manifest.json"
)

checks = {
    "protocol": (
        sha256_file(protocol_file)
        == EXPECTED_PROTOCOL_HASH
    ),
    "schema": (
        sha256_file(schema_file)
        == EXPECTED_SCHEMA_HASH
    ),
    "amendment_A1": (
        sha256_file(amendment_file)
        == EXPECTED_AMENDMENT_HASH
    ),
    "source_model_seal": (
        sha256_file(source_model_seal_file)
        == EXPECTED_SOURCE_MODEL_SEAL
    ),
}

if not all(checks.values()):
    raise RuntimeError(
        f"Artifact-chain verification failed: {checks}"
    )

with open(schema_file, "r", encoding="utf-8") as file:
    FEATURES = json.load(file)[
        "predictors_in_order"
    ]

print("=" * 78)
print("PAPER 3 — SOURCE-ONLY FEATURE-STABILITY ANALYSIS")
print("=" * 78)
print(f"Artifact-chain verification: {checks}")
print(f"Predictors: {len(FEATURES)}")
print(f"Permutation repeats: {N_REPEATS}")


warning_rows = []
stability_problems = []


for dataset in DATASETS:

    print(f"\n{'-' * 78}")
    print(f"DATASET: {dataset}")
    print(f"{'-' * 78}")

    for seed in SEEDS:

        seed_start = time.time()

        environment_file = (
            SANITIZED_DIR
            / dataset
            / f"seed_{seed}"
            / "environment.parquet"
        )

        environment_data = pd.read_parquet(
            environment_file
        )

        expected_features = set(FEATURES)

        if not expected_features.issubset(
            environment_data.columns
        ):
            raise RuntimeError(
                f"{dataset}, seed {seed}: "
                "missing environment predictors."
            )

        actual_counts = (
            environment_data.groupby([
                "actual_environment_id",
                "Label",
            ]).size()
        )

        pseudo_counts = (
            environment_data.groupby([
                "pseudo_environment_id",
                "Label",
            ]).size()
        )

        if not (actual_counts == 2000).all():
            raise RuntimeError(
                "Actual environment groups are not balanced."
            )

        if not (pseudo_counts == 2000).all():
            raise RuntimeError(
                "Pseudo-environment groups are not balanced."
            )

        preprocessor_file = (
            MODEL_DIR
            / dataset
            / f"seed_{seed}"
            / "source_preprocessor.joblib"
        )

        preprocessor_artifact = joblib.load(
            preprocessor_file
        )

        if (
            preprocessor_artifact[
                "protocol_sha256"
            ] != EXPECTED_PROTOCOL_HASH
        ):
            raise RuntimeError(
                "Preprocessor protocol mismatch."
            )

        X_environment_raw = environment_data[
            FEATURES
        ].to_numpy(dtype=np.float32)

        y_environment = environment_data[
            "Label"
        ].to_numpy(dtype=np.int8)

        X_environment = (
            preprocessor_artifact["imputer"]
            .transform(X_environment_raw)
            .astype(np.float32)
        )

        X_environment_scaled = (
            preprocessor_artifact[
                "logistic_regression_scaler"
            ]
            .transform(X_environment)
            .astype(np.float32)
        )

        group_definitions = []

        for group_id in sorted(
            environment_data[
                "actual_environment_id"
            ].unique()
        ):
            positions = np.flatnonzero(
                environment_data[
                    "actual_environment_id"
                ].to_numpy()
                == group_id
            )

            group_definitions.append({
                "group_type": "environmental",
                "group_id": str(group_id),
                "positions": positions,
            })

        for group_id in sorted(
            environment_data[
                "pseudo_environment_id"
            ].unique()
        ):
            positions = np.flatnonzero(
                environment_data[
                    "pseudo_environment_id"
                ].to_numpy()
                == group_id
            )

            group_definitions.append({
                "group_type": "sampling",
                "group_id": str(group_id),
                "positions": positions,
            })

        for model_name in MODELS:

            model_output_dir = (
                STABILITY_DIR
                / dataset
                / f"seed_{seed}"
                / model_name
            )

            model_output_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            completion_file = (
                model_output_dir
                / "stability_complete.json"
            )

            pfi_file = (
                model_output_dir
                / "permutation_importance.parquet"
            )

            pairwise_file = (
                model_output_dir
                / "pairwise_stability.parquet"
            )

            if completion_file.exists():
                with open(
                    completion_file,
                    "r",
                    encoding="utf-8",
                ) as file:
                    completed = json.load(file)

                if sha256_file(
                    pfi_file
                ) != completed[
                    "permutation_importance_sha256"
                ]:
                    raise RuntimeError(
                        "Completed PFI hash mismatch."
                    )

                if sha256_file(
                    pairwise_file
                ) != completed[
                    "pairwise_stability_sha256"
                ]:
                    raise RuntimeError(
                        "Completed pairwise hash mismatch."
                    )

                warning_rows.append(
                    completed["warning_scores"]
                )

                print(
                    f"  Seed {seed} | {model_name}: "
                    "SKIPPED—already completed"
                )
                continue

            source_model_completion_file = (
                MODEL_DIR
                / dataset
                / f"seed_{seed}"
                / "source_seed_complete.json"
            )

            with open(
                source_model_completion_file,
                "r",
                encoding="utf-8",
            ) as file:
                source_completion = json.load(file)

            model_record = source_completion[
                "models"
            ][model_name]

            model_file = Path(
                model_record[
                    "files"
                ]["model"]["path"]
            )

            if sha256_file(model_file) != model_record[
                "files"
            ]["model"]["sha256"]:
                raise RuntimeError(
                    "Source model artifact hash mismatch."
                )

            model_artifact = joblib.load(
                model_file
            )

            model = model_artifact["model"]
            threshold = float(
                model_artifact["threshold"]
            )

            uses_scaling = bool(
                model_artifact[
                    "uses_standard_scaler"
                ]
            )

            if uses_scaling:
                X_for_model = X_environment_scaled
            else:
                X_for_model = X_environment

            scorer = threshold_scorer(
                threshold
            )

            if pfi_file.exists():
                pfi_results = pd.read_parquet(
                    pfi_file
                )
            else:
                pfi_results = pd.DataFrame()

            completed_groups = set()

            if not pfi_results.empty:
                counts = (
                    pfi_results.groupby([
                        "group_type",
                        "group_id",
                    ]).size()
                )

                completed_groups = {
                    (
                        str(group_type),
                        str(group_id),
                    )
                    for (
                        group_type,
                        group_id,
                    ), count in counts.items()
                    if count == len(FEATURES)
                }

                incomplete_groups = {
                    (
                        str(group_type),
                        str(group_id),
                    )
                    for (
                        group_type,
                        group_id,
                    ), count in counts.items()
                    if count != len(FEATURES)
                }

                if incomplete_groups:
                    remove_mask = pfi_results.apply(
                        lambda row: (
                            str(row["group_type"]),
                            str(row["group_id"]),
                        ) in incomplete_groups,
                        axis=1,
                    )

                    pfi_results = pfi_results[
                        ~remove_mask
                    ].reset_index(drop=True)

            groups_finished_now = 0
            total_groups = len(group_definitions)

            for group in group_definitions:

                group_key = (
                    group["group_type"],
                    str(group["group_id"]),
                )

                if group_key in completed_groups:
                    continue

                positions = group["positions"]

                X_group = X_for_model[positions]
                y_group = y_environment[positions]

                label_counts = np.bincount(
                    y_group,
                    minlength=2,
                )

                if not np.array_equal(
                    label_counts,
                    np.array([2000, 2000]),
                ):
                    raise RuntimeError(
                        f"Unbalanced PFI group: {group_key}"
                    )

                baseline_ba = float(
                    scorer(
                        model,
                        X_group,
                        y_group,
                    )
                )

                random_state = stable_random_state(
                    dataset,
                    seed,
                    model_name,
                    group["group_type"],
                    group["group_id"],
                    "PFI",
                )

                pfi = permutation_importance(
                    estimator=model,
                    X=X_group,
                    y=y_group,
                    scoring=scorer,
                    n_repeats=N_REPEATS,
                    random_state=random_state,
                    n_jobs=1,
                )

                group_rows = []

                for feature_index, feature in enumerate(
                    FEATURES
                ):
                    group_rows.append({
                        "dataset": dataset,
                        "seed": seed,
                        "model": model_name,
                        "group_type": (
                            group["group_type"]
                        ),
                        "group_id": str(
                            group["group_id"]
                        ),
                        "feature": feature,
                        "baseline_balanced_accuracy": (
                            baseline_ba
                        ),
                        "importance_mean": float(
                            pfi.importances_mean[
                                feature_index
                            ]
                        ),
                        "importance_std": float(
                            pfi.importances_std[
                                feature_index
                            ]
                        ),
                        "repeat_1": float(
                            pfi.importances[
                                feature_index,
                                0,
                            ]
                        ),
                        "repeat_2": float(
                            pfi.importances[
                                feature_index,
                                1,
                            ]
                        ),
                        "repeat_3": float(
                            pfi.importances[
                                feature_index,
                                2,
                            ]
                        ),
                        "rows": int(len(y_group)),
                        "benign_rows": 2000,
                        "attack_rows": 2000,
                        "permutation_random_state": (
                            random_state
                        ),
                    })

                group_frame = pd.DataFrame(
                    group_rows
                )

                pfi_results = pd.concat(
                    [
                        pfi_results,
                        group_frame,
                    ],
                    ignore_index=True,
                )

                atomic_parquet(
                    pfi_results,
                    pfi_file,
                )

                completed_groups.add(group_key)
                groups_finished_now += 1

                if (
                    groups_finished_now % 5 == 0
                    or len(completed_groups)
                    == total_groups
                ):
                    print(
                        f"    Seed {seed} | {model_name}: "
                        f"{len(completed_groups)}/"
                        f"{total_groups} groups completed"
                    )

            expected_result_rows = (
                total_groups * len(FEATURES)
            )

            if len(pfi_results) != expected_result_rows:
                raise RuntimeError(
                    f"Unexpected PFI row count: "
                    f"{len(pfi_results)} != "
                    f"{expected_result_rows}"
                )

            environmental_pairs, environmental_summary = (
                calculate_pairwise_stability(
                    pfi_results,
                    FEATURES,
                    "environmental",
                )
            )

            sampling_pairs, sampling_summary = (
                calculate_pairwise_stability(
                    pfi_results,
                    FEATURES,
                    "sampling",
                )
            )

            environmental_pairs[
                "dataset"
            ] = dataset

            environmental_pairs["seed"] = seed

            environmental_pairs[
                "model"
            ] = model_name

            sampling_pairs["dataset"] = dataset
            sampling_pairs["seed"] = seed
            sampling_pairs["model"] = model_name

            pairwise_results = pd.concat(
                [
                    environmental_pairs,
                    sampling_pairs,
                ],
                ignore_index=True,
            )

            atomic_parquet(
                pairwise_results,
                pairwise_file,
            )

            environmental_stability = float(
                environmental_summary[
                    "mean_spearman"
                ]
            )

            sampling_stability = float(
                sampling_summary[
                    "mean_spearman"
                ]
            )

            warning_scores = {
                "dataset": dataset,
                "seed": seed,
                "model": model_name,
                "environmental_stability": (
                    environmental_stability
                ),
                "sampling_stability": (
                    sampling_stability
                ),
                "environmental_instability": float(
                    1.0
                    - environmental_stability
                ),
                "sampling_environment_gap": float(
                    sampling_stability
                    - environmental_stability
                ),
                "environmental_top10_jaccard": float(
                    environmental_summary[
                        "mean_top10_jaccard"
                    ]
                ),
                "sampling_top10_jaccard": float(
                    sampling_summary[
                        "mean_top10_jaccard"
                    ]
                ),
                "environmental_groups": int(
                    environmental_summary["groups"]
                ),
                "sampling_groups": int(
                    sampling_summary["groups"]
                ),
                "environmental_pairs": int(
                    environmental_summary["pairs"]
                ),
                "sampling_pairs": int(
                    sampling_summary["pairs"]
                ),
                "undefined_environmental_pairs": int(
                    environmental_summary[
                        "undefined_spearman_pairs"
                    ]
                ),
                "undefined_sampling_pairs": int(
                    sampling_summary[
                        "undefined_spearman_pairs"
                    ]
                ),
                "mean_environmental_baseline_ba": float(
                    pfi_results[
                        pfi_results["group_type"]
                        == "environmental"
                    ][
                        "baseline_balanced_accuracy"
                    ].mean()
                ),
                "mean_sampling_baseline_ba": float(
                    pfi_results[
                        pfi_results["group_type"]
                        == "sampling"
                    ][
                        "baseline_balanced_accuracy"
                    ].mean()
                ),
            }

            stability_gate_pass = (
                np.isfinite(
                    list(
                        warning_scores.values()
                    )[3:]
                ).all()
                and warning_scores[
                    "environmental_groups"
                ] >= 4
                and warning_scores[
                    "sampling_groups"
                ] >= 4
            )

            if not stability_gate_pass:
                stability_problems.append(
                    f"{dataset}, seed {seed}, "
                    f"{model_name}: stability gate failed"
                )

            completion = {
                "dataset": dataset,
                "seed": seed,
                "model": model_name,
                "created_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "protocol_sha256": (
                    EXPECTED_PROTOCOL_HASH
                ),
                "amendment_A1_sha256": (
                    EXPECTED_AMENDMENT_HASH
                ),
                "feature_schema_sha256": (
                    EXPECTED_SCHEMA_HASH
                ),
                "source_model_seal_sha256": (
                    EXPECTED_SOURCE_MODEL_SEAL
                ),
                "threshold": threshold,
                "permutation_repeats": N_REPEATS,
                "features": len(FEATURES),
                "environmental_summary": (
                    environmental_summary
                ),
                "sampling_summary": sampling_summary,
                "warning_scores": warning_scores,
                "permutation_importance_sha256": (
                    sha256_file(pfi_file)
                ),
                "pairwise_stability_sha256": (
                    sha256_file(pairwise_file)
                ),
                "stability_gate_pass": (
                    stability_gate_pass
                ),
                "status": (
                    "PASS"
                    if stability_gate_pass
                    else "FAIL"
                ),
            }

            completion_file.write_text(
                json.dumps(
                    completion,
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )

            warning_rows.append(warning_scores)

            print(
                f"  Seed {seed} | {model_name}: "
                f"S_env={environmental_stability:.4f}, "
                f"S_sampling={sampling_stability:.4f}, "
                f"gap="
                f"{warning_scores['sampling_environment_gap']:.4f}, "
                f"env-top10="
                f"{warning_scores['environmental_top10_jaccard']:.4f}, "
                f"gate="
                f"{'PASS' if stability_gate_pass else 'FAIL'}"
            )

        print(
            f"  Seed {seed} stability elapsed: "
            f"{(time.time() - seed_start) / 60:.1f} min"
        )


# ----------------------------------------------------------------
# Seal all source-only warning scores before target evaluation
# ----------------------------------------------------------------

warning_table = pd.DataFrame(warning_rows)

warning_table = warning_table.sort_values([
    "dataset",
    "seed",
    "model",
]).reset_index(drop=True)

if len(warning_table) != 60:
    raise RuntimeError(
        f"Expected 60 warning rows, found "
        f"{len(warning_table)}."
    )

warning_file = (
    STABILITY_DIR
    / "source_only_warning_scores.csv"
)

warning_table.to_csv(
    warning_file,
    index=False,
)

seal_entries = []

for dataset in DATASETS:
    for seed in SEEDS:
        for model_name in MODELS:
            model_output_dir = (
                STABILITY_DIR
                / dataset
                / f"seed_{seed}"
                / model_name
            )

            for filename in [
                "stability_complete.json",
                "permutation_importance.parquet",
                "pairwise_stability.parquet",
            ]:
                path = model_output_dir / filename

                if not path.exists():
                    raise RuntimeError(
                        f"Missing stability artifact: {path}"
                    )

                seal_entries.append({
                    "relative_path": str(
                        path.relative_to(
                            STABILITY_DIR
                        )
                    ),
                    "sha256": sha256_file(path),
                })

seal_entries.append({
    "relative_path": warning_file.name,
    "sha256": sha256_file(warning_file),
})

stability_seal = {
    "seal_id": "P3_SOURCE_ONLY_STABILITY_SEAL_V1",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "protocol_sha256": EXPECTED_PROTOCOL_HASH,
    "amendment_A1_sha256": EXPECTED_AMENDMENT_HASH,
    "feature_schema_sha256": EXPECTED_SCHEMA_HASH,
    "source_model_seal_sha256": (
        EXPECTED_SOURCE_MODEL_SEAL
    ),
    "warning_score_rows": int(
        len(warning_table)
    ),
    "target_datasets_evaluated": 0,
    "target_results_seen": False,
    "stability_problems": stability_problems,
    "artifacts": sorted(
        seal_entries,
        key=lambda item: item["relative_path"],
    ),
    "status": (
        "PASS"
        if not stability_problems
        else "FAIL"
    ),
}

stability_seal_file = (
    STABILITY_DIR
    / "source_only_stability_seal_manifest.json"
)

stability_seal_file.write_text(
    json.dumps(
        stability_seal,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

stability_seal_hash = sha256_file(
    stability_seal_file
)

(
    STABILITY_DIR
    / "source_only_stability_seal_manifest.sha256"
).write_text(
    stability_seal_hash,
    encoding="utf-8",
)

print("\n" + "=" * 78)
print(
    "SOURCE-ONLY FEATURE-STABILITY SEAL: "
    f"{stability_seal['status']}"
)
print(f"Warning-score rows: {len(warning_table)}")
print(f"Problems: {stability_problems}")
print(
    "Source-only stability seal SHA-256: "
    f"{stability_seal_hash}"
)
print(f"Warning scores: {warning_file}")
print("=" * 78)

if stability_seal["status"] != "PASS":
    raise RuntimeError(
        "Source-only stability seal failed. "
        "Do not evaluate targets."
    )

PAPER 3 — SOURCE-ONLY FEATURE-STABILITY ANALYSIS
Artifact-chain verification: {'protocol': True, 'schema': True, 'amendment_A1': True, 'source_model_seal': True}
Predictors: 49
Permutation repeats: 3

------------------------------------------------------------------------------
DATASET: NF_BoT_IoT_v3
------------------------------------------------------------------------------
    Seed 2027 | LogisticRegression: 5/10 groups completed
    Seed 2027 | LogisticRegression: 10/10 groups completed
  Seed 2027 | LogisticRegression: S_env=0.5450, S_sampling=0.9305, gap=0.3855, env-top10=0.5847, gate=PASS
    Seed 2027 | XGBoost: 5/10 groups completed
    Seed 2027 | XGBoost: 10/10 groups completed
  Seed 2027 | XGBoost: S_env=0.3143, S_sampling=0.8141, gap=0.4999, env-top10=0.4068, gate=PASS
    Seed 2027 | RandomForest: 5/10 groups completed
    Seed 2027 | RandomForest: 10/10 groups completed
  Seed 2027 | RandomForest: S_env=0.3388, S_sampling=0.7381, gap=0.3993, env-top10=0.4886, gate=PA

In [12]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
from itertools import permutations

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

PROTOCOL_DIR = ROOT / "Paper3_Protocol"
MATERIALIZED_DIR = ROOT / "Paper3_Materialized"
MODEL_DIR = ROOT / "Paper3_Source_Models"
STABILITY_DIR = ROOT / "Paper3_Source_Stability"

EXPECTED_HASHES = {
    "protocol": (
        "e39a5007a7f2cbdc348d5179ce593af"
        "5766bc6688f5baa3a46a56399f683a488"
    ),
    "schema": (
        "67d61eebb3377e3053f943f1e4c03123"
        "7b3ef35dd06296d8538fe552b16be0c6"
    ),
    "amendment_A1": (
        "19c30a37d62d36f5b3c9bb795f64e105"
        "99dcef78a9c51bf96ca80b58d7db4b47"
    ),
    "source_model_seal": (
        "81eebf7b013fec0ca1802ebb54894fe73"
        "d7023b2c6450da09c72af3b3f5a3182"
    ),
    "source_stability_seal": (
        "3ec5641ed96a462ce2c76765b6b4483a"
        "1e48cad11d79981feb7afd1c43449340"
    ),
}

ARTIFACTS = {
    "protocol": (
        PROTOCOL_DIR
        / "paper3_protocol_frozen_v1.json"
    ),
    "schema": (
        MATERIALIZED_DIR
        / "feature_schema_v1.json"
    ),
    "amendment_A1": (
        PROTOCOL_DIR
        / "amendment_A1_duplicate_group_disjoint.json"
    ),
    "source_model_seal": (
        MODEL_DIR
        / "source_model_seal_manifest.json"
    ),
    "source_stability_seal": (
        STABILITY_DIR
        / "source_only_stability_seal_manifest.json"
    ),
}

DATASETS = [
    "NF_BoT_IoT_v3",
    "NF_CSE_CIC_IDS2018_v3",
    "NF_ToN_IoT_v3",
    "NF_UNSW_NB15_v3",
]


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


checks = {
    name: sha256_file(path)
    == EXPECTED_HASHES[name]
    for name, path in ARTIFACTS.items()
}

print("=" * 76)
print("PAPER 3 — TARGET-EVALUATION PLAN FREEZE")
print("=" * 76)
print(f"Artifact-chain verification: {checks}")

if not all(checks.values()):
    raise RuntimeError(
        "Artifact-chain verification failed."
    )

directions = [
    {
        "source": source,
        "target": target,
    }
    for source, target in permutations(
        DATASETS,
        2,
    )
]

amendment_file = (
    PROTOCOL_DIR
    / "amendment_A2_target_evaluation_plan.json"
)

hash_file = (
    PROTOCOL_DIR
    / "amendment_A2_target_evaluation_plan.sha256"
)

if amendment_file.exists():
    amendment_hash = sha256_file(
        amendment_file
    )

    recorded_hash = (
        hash_file.read_text(
            encoding="utf-8"
        ).strip()
    )

    if amendment_hash != recorded_hash:
        raise RuntimeError(
            "Existing Amendment A2 hash mismatch."
        )

else:
    amendment = {
        "amendment_id": (
            "P3_A2_TARGET_EVALUATION_PLAN"
        ),
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "timing": (
            "Frozen after all source-only warning scores "
            "were cryptographically sealed and before any "
            "Paper 3 cross-dataset target performance was "
            "calculated."
        ),
        "parent_protocol_sha256": (
            EXPECTED_HASHES["protocol"]
        ),
        "amendment_A1_sha256": (
            EXPECTED_HASHES["amendment_A1"]
        ),
        "feature_schema_sha256": (
            EXPECTED_HASHES["schema"]
        ),
        "source_model_seal_sha256": (
            EXPECTED_HASHES[
                "source_model_seal"
            ]
        ),
        "source_stability_seal_sha256": (
            EXPECTED_HASHES[
                "source_stability_seal"
            ]
        ),
        "evaluation_design": {
            "datasets": DATASETS,
            "directed_transfers": directions,
            "number_of_directions": 12,
            "source_seeds": [
                2027,
                2028,
                2029,
                2030,
                2031,
            ],
            "models": [
                "LogisticRegression",
                "RandomForest",
                "XGBoost",
            ],
            "prediction_jobs": 180,
        },
        "target_candidate_panel": {
            "selection_seed": 42027,
            "candidate_rows_per_class": 50000,
            "total_candidate_rows_per_target": (
                100000
            ),
            "selection": (
                "Deterministic lowest row-hash priority "
                "within each target label."
            ),
            "replacement": False,
        },
        "direction_specific_panel": {
            "final_rows_per_class": 40000,
            "final_rows_per_direction_seed": (
                80000
            ),
            "shared_across_models": True,
            "cross_dataset_duplicate_rule": (
                "Exclude target feature signatures matching "
                "any source train, calibration, source-test, "
                "or environmental signature for that source "
                "dataset and seed."
            ),
            "insufficient_rows_rule": (
                "Hard stop if fewer than 40000 rows remain "
                "for either class; do not reduce the quota "
                "after viewing target outcomes."
            ),
        },
        "target_processing": {
            "predictors": 49,
            "target_fitted_imputation": False,
            "target_fitted_scaling": False,
            "target_threshold_calibration": False,
            "use_frozen_source_imputer": True,
            "use_frozen_source_scaler_for_LR": True,
            "use_frozen_source_threshold": True,
            "target_adaptation": False,
        },
        "label_separation": {
            "candidate_features_and_labels_saved_separately": True,
            "prediction_code_must_not_read_label_vault": True,
            "predictions_sealed_before_label_join": True,
            "target_labels_used_for": (
                "Balanced evaluation-panel construction "
                "and final evaluation only."
            ),
        },
        "primary_outcome": {
            "metric": "balanced_accuracy_loss",
            "formula": (
                "source_test_balanced_accuracy "
                "- target_balanced_accuracy"
            ),
        },
        "secondary_outcomes": [
            "target balanced accuracy",
            "target Macro F1",
            "target MCC",
            "target PR-AUC",
            "benign recall",
            "attack recall",
        ],
        "primary_warning_predictors": [
            "environmental_instability",
            "sampling_environment_gap",
        ],
        "baseline_warning_predictor": (
            "sampling instability"
        ),
        "confirmatory_rules": {
            "all_60_warning_scores_already_sealed": True,
            "no_model_selection_using_target_results": True,
            "no_threshold_change_using_target_results": True,
            "all_180_prediction_results_reported": True,
            "negative_results_reported": True,
        },
        "paper3_target_results_seen_before_freeze": False,
    }

    amendment_text = json.dumps(
        amendment,
        indent=2,
        sort_keys=True,
    )

    amendment_file.write_text(
        amendment_text,
        encoding="utf-8",
    )

    amendment_hash = sha256_file(
        amendment_file
    )

    hash_file.write_text(
        amendment_hash,
        encoding="utf-8",
    )

print("Target-evaluation plan: FROZEN")
print(f"Directed transfers: {len(directions)}")
print("Prediction jobs: 180")
print(f"Amendment A2 SHA-256: {amendment_hash}")
print("Paper 3 target results previously calculated: NO")
print("=" * 76)

PAPER 3 — TARGET-EVALUATION PLAN FREEZE
Artifact-chain verification: {'protocol': True, 'schema': True, 'amendment_A1': True, 'source_model_seal': True, 'source_stability_seal': True}
Target-evaluation plan: FROZEN
Directed transfers: 12
Prediction jobs: 180
Amendment A2 SHA-256: 2217078acfa10700f4fae29c6f32f171ce7a4e2ecb22d22ce5eabb6087e5aa5e
Paper 3 target results previously calculated: NO


In [13]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

EXPECTED_A2 = (
    "2217078acfa10700f4fae29c6f32f171"
    "ce7a4e2ecb22d22ce5eabb6087e5aa5e"
)

A2_PATH = (
    ROOT / "Paper3_Protocol" /
    "amendment_A2_target_evaluation_plan.json"
)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

print("=" * 78)
print("PAPER 3 — TARGET-STAGE READ-ONLY PREFLIGHT")
print("=" * 78)

assert A2_PATH.exists(), f"Missing A2: {A2_PATH}"

observed_a2 = sha256_file(A2_PATH)
print("A2 path:", A2_PATH)
print("Expected A2 SHA-256:", EXPECTED_A2)
print("Observed A2 SHA-256:", observed_a2)
print("A2 verification:", "PASS" if observed_a2 == EXPECTED_A2 else "FAIL")

assert observed_a2 == EXPECTED_A2, "A2 hash mismatch"

with open(A2_PATH, "r", encoding="utf-8") as f:
    a2 = json.load(f)

print("\nA2 top-level keys:")
print(sorted(a2.keys()))

print("\nRelevant directories:")
for name in [
    "Paper3_Protocol",
    "Paper3_Materialized",
    "Paper3_Sanitized",
    "Paper3_Source_Models",
    "Paper3_Source_Stability",
]:
    path = ROOT / name
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

sanitized_root = ROOT / "Paper3_Sanitized"
parquet_files = sorted(sanitized_root.rglob("*.parquet"))
csv_files = sorted(sanitized_root.rglob("*.csv"))
json_files = sorted(sanitized_root.rglob("*.json"))

print("\nSanitized inventory:")
print("Parquet files:", len(parquet_files))
print("CSV files:", len(csv_files))
print("JSON files:", len(json_files))

print("\nSanitized Parquet paths:")
for path in parquet_files:
    try:
        frame = pd.read_parquet(path)
        print("-" * 78)
        print("PATH:", path.relative_to(ROOT))
        print("ROWS:", len(frame))
        print("COLUMNS:", list(frame.columns))
    except Exception as exc:
        print("READ FAILURE:", path.relative_to(ROOT), repr(exc))

print("\nSanitized CSV paths:")
for path in csv_files:
    print(" ", path.relative_to(ROOT))

print("\nSanitized JSON paths:")
for path in json_files:
    print(" ", path.relative_to(ROOT))

print("\nSource metrics schema:")
source_metrics = (
    ROOT / "Paper3_Source_Models" /
    "source_test_metrics_all_models.csv"
)
if source_metrics.exists():
    df = pd.read_csv(source_metrics, nrows=5)
    print("PATH:", source_metrics)
    print("COLUMNS:", list(df.columns))
    print(df.to_string(index=False))
else:
    print("MISSING:", source_metrics)

print("\nWarning-score schema:")
warning_scores = (
    ROOT / "Paper3_Source_Stability" /
    "source_only_warning_scores.csv"
)
if warning_scores.exists():
    df = pd.read_csv(warning_scores, nrows=5)
    print("PATH:", warning_scores)
    print("COLUMNS:", list(df.columns))
    print(df.to_string(index=False))
else:
    print("MISSING:", warning_scores)

print("\nTARGET-STAGE READ-ONLY PREFLIGHT: PASS")
print("No target panels, predictions, or performance results were created.")
print("=" * 78)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER 3 — TARGET-STAGE READ-ONLY PREFLIGHT
A2 path: /content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet/Paper3_Protocol/amendment_A2_target_evaluation_plan.json
Expected A2 SHA-256: 2217078acfa10700f4fae29c6f32f171ce7a4e2ecb22d22ce5eabb6087e5aa5e
Observed A2 SHA-256: 2217078acfa10700f4fae29c6f32f171ce7a4e2ecb22d22ce5eabb6087e5aa5e
A2 verification: PASS

A2 top-level keys:
['amendment_A1_sha256', 'amendment_id', 'baseline_warning_predictor', 'confirmatory_rules', 'created_utc', 'direction_specific_panel', 'evaluation_design', 'feature_schema_sha256', 'label_separation', 'paper3_target_results_seen_before_freeze', 'parent_protocol_sha256', 'primary_outcome', 'primary_warning_predictors', 'secondary_outcomes', 'source_model_seal_sha256', 'source_stability_seal_sha256', 'target_candidate_panel', 'target_processing', 'timing']

Relevant direc

In [14]:
import runpy
from pathlib import Path

script_path = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet/Paper3_Code/"
    "paper3_target_candidate_panels.py"
)

assert script_path.exists(), f"Missing: {script_path}"
runpy.run_path(str(script_path), run_name="__main__")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER 3 — TARGET CANDIDATE PANELS AND LABEL-VAULT SEAL
Expected A2 SHA-256: 2217078acfa10700f4fae29c6f32f171ce7a4e2ecb22d22ce5eabb6087e5aa5e
Observed A2 SHA-256: 2217078acfa10700f4fae29c6f32f171ce7a4e2ecb22d22ce5eabb6087e5aa5e
A2 verification: PASS
Feature schema verification: PASS
Predictors: 49
Candidate size per target: 100,000
Target labels are written only to the separate label vault.

------------------------------------------------------------------------------
TARGET DATASET: NF_BoT_IoT_v3
------------------------------------------------------------------------------
    NF_BoT_IoT_v3: chunks=10, rows=2,500,000, kept0=281, kept1=50,000
    NF_BoT_IoT_v3: chunks=20, rows=5,000,000, kept0=916, kept1=50,000
    NF_BoT_IoT_v3: chunks=30, rows=7,500,000, kept0=1,852, kept1=50,000
    NF_BoT_IoT_v3: chunks=40, rows=10,000,000, kept0=1,852, kept1=50,000
    

{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet/Paper3_Code/paper3_target_candidate_panels.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import

In [15]:
import runpy
from pathlib import Path

script_path = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet/Paper3_Code/"
    "paper3_direction_indices.py"
)

assert script_path.exists(), f"Missing: {script_path}"

_ = runpy.run_path(str(script_path), run_name="__main__")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER 3 — DIRECTION-SPECIFIC TARGET-INDEX CONSTRUCTION
A2 verification: PASS
Target candidate-panel seal verification: PASS
Candidate-panel seal SHA-256: 084427a57c884ccb4ecdc4298975e5ff59f4ca632ecdedf24c62edb82c4b2df1
Predictors: 49
Directed transfers: 12
Direction-specific indices: 60
Rows per final index: 80,000
Target performance calculated before construction: NO

Loading four candidate feature panels and the separated construction labels...
  NF_BoT_IoT_v3: candidate verification and exact signatures — PASS
  NF_CSE_CIC_IDS2018_v3: candidate verification and exact signatures — PASS
  NF_ToN_IoT_v3: candidate verification and exact signatures — PASS
  NF_UNSW_NB15_v3: candidate verification and exact signatures — PASS

----------------------------------------------------------------------------------
SOURCE DATASET: NF_BoT_IoT_v3
------------------------

In [16]:
from pathlib import Path
import json

ROOT = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet"
)

MODEL_ROOT = ROOT / "Paper3_Source_Models"

print("=" * 80)
print("PAPER 3 — SOURCE-MODEL ARTIFACT INVENTORY")
print("=" * 80)

assert MODEL_ROOT.exists(), f"Missing: {MODEL_ROOT}"

files = sorted(path for path in MODEL_ROOT.rglob("*") if path.is_file())

print("Total files:", len(files))

for path in files:
    relative = path.relative_to(MODEL_ROOT)
    print(f"{relative} | {path.stat().st_size:,} bytes")

print("\nJSON structures:")
for path in sorted(MODEL_ROOT.rglob("*.json")):
    print("\n" + "-" * 80)
    print("JSON:", path.relative_to(MODEL_ROOT))
    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            print("Top-level keys:", sorted(obj.keys()))

            for key, value in obj.items():
                key_lower = key.lower()
                if any(word in key_lower for word in [
                    "model", "imputer", "scaler", "threshold",
                    "feature", "path", "file", "hash"
                ]):
                    print(f"{key}: {value}")
        else:
            print("JSON type:", type(obj).__name__)
    except Exception as exc:
        print("READ FAILURE:", repr(exc))

print("\nSOURCE-MODEL ARTIFACT INVENTORY: PASS")
print("No model was loaded.")
print("No target feature or label was read.")
print("No prediction or performance result was calculated.")
print("=" * 80)

PAPER 3 — SOURCE-MODEL ARTIFACT INVENTORY
Total files: 223
NF_BoT_IoT_v3/seed_2027/LogisticRegression_calibration_predictions.parquet | 116,612 bytes
NF_BoT_IoT_v3/seed_2027/LogisticRegression_model.joblib | 1,724 bytes
NF_BoT_IoT_v3/seed_2027/LogisticRegression_source_test_predictions.parquet | 120,044 bytes
NF_BoT_IoT_v3/seed_2027/RandomForest_calibration_predictions.parquet | 53,814 bytes
NF_BoT_IoT_v3/seed_2027/RandomForest_model.joblib | 3,494,014 bytes
NF_BoT_IoT_v3/seed_2027/RandomForest_source_test_predictions.parquet | 54,733 bytes
NF_BoT_IoT_v3/seed_2027/XGBoost_calibration_predictions.parquet | 59,965 bytes
NF_BoT_IoT_v3/seed_2027/XGBoost_model.joblib | 453,500 bytes
NF_BoT_IoT_v3/seed_2027/XGBoost_source_test_predictions.parquet | 61,248 bytes
NF_BoT_IoT_v3/seed_2027/source_preprocessor.joblib | 2,553 bytes
NF_BoT_IoT_v3/seed_2027/source_seed_complete.json | 8,070 bytes
NF_BoT_IoT_v3/seed_2028/LogisticRegression_calibration_predictions.parquet | 117,054 bytes
NF_BoT_IoT_v3/

In [17]:
from pathlib import Path
import joblib

path = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet/Paper3_Source_Models/"
    "NF_BoT_IoT_v3/seed_2027/source_preprocessor.joblib"
)

assert path.exists(), f"Missing: {path}"

obj = joblib.load(path)

print("=" * 70)
print("SOURCE PREPROCESSOR STRUCTURE")
print("=" * 70)
print("Python type:", type(obj))
print("Representation:", repr(obj))

if isinstance(obj, dict):
    print("Dictionary keys:", sorted(obj.keys()))
    for key, value in obj.items():
        print(f"{key}: {type(value)}")
elif hasattr(obj, "named_steps"):
    print("Pipeline steps:", obj.named_steps)
elif hasattr(obj, "__dict__"):
    print("Attributes:", sorted(obj.__dict__.keys()))
    for key, value in obj.__dict__.items():
        print(f"{key}: {type(value)}")

print("\nNo target feature or label was accessed.")
print("No prediction was calculated.")

SOURCE PREPROCESSOR STRUCTURE
Python type: <class 'dict'>
Representation: {'dataset': 'NF_BoT_IoT_v3', 'seed': 2027, 'features': ['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC

In [18]:
import runpy
from pathlib import Path

script_path = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet/Paper3_Code/"
    "paper3_label_blind_target_predictions.py"
)

assert script_path.exists(), f"Missing: {script_path}"

_ = runpy.run_path(str(script_path), run_name="__main__")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER 3 — LABEL-BLIND CROSS-DATASET TARGET PREDICTIONS
Amendment A2 verification: PASS
Source-model seal verification: PASS
Target candidate-panel seal verification: PASS
Direction-index seal verification: PASS
Prediction jobs: 180
Rows per job: 80,000
Target label-vault path present in this program: NO
Target label rows read before prediction: 0
Software versions: {'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit_learn': '1.6.1', 'xgboost': '3.4.1', 'joblib': '1.5.3'}

Loading prediction-visible candidate feature panels...
  NF_BoT_IoT_v3: rows=100,000, target-label-columns=0 — PASS
  NF_CSE_CIC_IDS2018_v3: rows=100,000, target-label-columns=0 — PASS
  NF_ToN_IoT_v3: rows=100,000, target-label-columns=0 — PASS
  NF_UNSW_NB15_v3: rows=100,000, target-label-columns=0 — PASS

---------------------------------------------------------------------

AssertionError: 

In [19]:
from pathlib import Path
import joblib

model_dir = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet/Paper3_Source_Models/"
    "NF_BoT_IoT_v3/seed_2027"
)

models = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
]

print("=" * 75)
print("SOURCE MODEL OBJECT INSPECTION")
print("=" * 75)

for model_name in models:
    path = model_dir / f"{model_name}_model.joblib"
    obj = joblib.load(path)

    print("\nModel:", model_name)
    print("Type:", type(obj))
    print("Has predict_proba:", hasattr(obj, "predict_proba"))
    print("Has predict:", hasattr(obj, "predict"))

    if isinstance(obj, dict):
        print("Dictionary keys:", sorted(obj.keys()))
        for key, value in obj.items():
            print(f"  {key}: {type(value)}")

    if hasattr(obj, "get_booster"):
        print("Has get_booster: True")

    if hasattr(obj, "num_features"):
        try:
            print("Number of features:", obj.num_features())
        except Exception:
            pass

print("\nNo target data or labels were accessed.")
print("No prediction was calculated.")

SOURCE MODEL OBJECT INSPECTION

Model: LogisticRegression
Type: <class 'dict'>
Has predict_proba: False
Has predict: False
Dictionary keys: ['amendment_A1_sha256', 'dataset', 'feature_schema_sha256', 'features', 'model', 'model_name', 'protocol_sha256', 'seed', 'threshold', 'uses_standard_scaler']
  dataset: <class 'str'>
  seed: <class 'int'>
  model_name: <class 'str'>
  model: <class 'sklearn.linear_model._logistic.LogisticRegression'>
  uses_standard_scaler: <class 'bool'>
  threshold: <class 'float'>
  features: <class 'list'>
  protocol_sha256: <class 'str'>
  amendment_A1_sha256: <class 'str'>
  feature_schema_sha256: <class 'str'>

Model: RandomForest
Type: <class 'dict'>
Has predict_proba: False
Has predict: False
Dictionary keys: ['amendment_A1_sha256', 'dataset', 'feature_schema_sha256', 'features', 'model', 'model_name', 'protocol_sha256', 'seed', 'threshold', 'uses_standard_scaler']
  dataset: <class 'str'>
  seed: <class 'int'>
  model_name: <class 'str'>
  model: <class 

In [20]:
from pathlib import Path
import hashlib
import runpy

script_path = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet/Paper3_Code/"
    "paper3_label_blind_target_predictions.py"
)

expected_sha256 = (
    "bdf16dab915ea5994d9917b89f6902fcf20905ee8a07e5991eeda45a300265cb"
)

observed_sha256 = hashlib.sha256(script_path.read_bytes()).hexdigest()

print("Observed SHA-256:", observed_sha256)
assert observed_sha256 == expected_sha256, "Old or altered script detected"

_ = runpy.run_path(str(script_path), run_name="__main__")

Observed SHA-256: bdf16dab915ea5994d9917b89f6902fcf20905ee8a07e5991eeda45a300265cb
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER 3 — LABEL-BLIND CROSS-DATASET TARGET PREDICTIONS
Amendment A2 verification: PASS
Source-model seal verification: PASS
Target candidate-panel seal verification: PASS
Direction-index seal verification: PASS
Prediction jobs: 180
Rows per job: 80,000
Target label-vault path present in this program: NO
Target label rows read before prediction: 0
Software versions: {'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit_learn': '1.6.1', 'xgboost': '3.4.1', 'joblib': '1.5.3'}

Loading prediction-visible candidate feature panels...
  NF_BoT_IoT_v3: rows=100,000, target-label-columns=0 — PASS
  NF_CSE_CIC_IDS2018_v3: rows=100,000, target-label-columns=0 — PASS
  NF_ToN_IoT_v3: rows=100,000, target-label-columns=0 — PASS
  NF_UNSW_NB15_v3: rows=100,000, target-label-colu

In [21]:
from pathlib import Path
import hashlib
import runpy

script_path = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet/Paper3_Code/"
    "paper3_final_target_evaluation.py"
)

expected_sha256 = (
    "ef9b7b01a25a1efa72160bbf4a03f264"
    "a391b20fa52b9e87fc9d59a4d028734e"
)

observed_sha256 = hashlib.sha256(script_path.read_bytes()).hexdigest()
print("Observed SHA-256:", observed_sha256)

assert observed_sha256 == expected_sha256, "Wrong or altered script"
_ = runpy.run_path(str(script_path), run_name="__main__")

Observed SHA-256: ef9b7b01a25a1efa72160bbf4a03f264a391b20fa52b9e87fc9d59a4d028734e
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER 3 — FINAL TARGET EVALUATION AND CONFIRMATORY ANALYSIS
Amendment A2 verification: PASS
Source-model and source-stability chain: PASS
All 180 sealed target predictions verified before label access: PASS
Target label rows read so far: 0
Amendment A3 confirmatory-analysis seal SHA-256: dd48b05318d8369ece3a98ca341774c30d7521e2aa655f1742271b104fca148b
Amendment A3 timing: FROZEN BEFORE TARGET LABEL JOIN
Target label rows read: 0
Target performance calculated: NO

FIRST-RUN GATE: PASS
Run this same script one more time to perform the authorized label reveal.


In [22]:
from pathlib import Path
import hashlib
import runpy

script_path = Path(
    "/content/drive/MyDrive/research/researchdata/"
    "NIDS_Research/Paper3_DataSet/Paper3_Code/"
    "paper3_final_target_evaluation.py"
)

expected_sha256 = (
    "ef9b7b01a25a1efa72160bbf4a03f264"
    "a391b20fa52b9e87fc9d59a4d028734e"
)

observed_sha256 = hashlib.sha256(script_path.read_bytes()).hexdigest()
print("Observed SHA-256:", observed_sha256)

assert observed_sha256 == expected_sha256, "Wrong or altered script"
_ = runpy.run_path(str(script_path), run_name="__main__")

Observed SHA-256: ef9b7b01a25a1efa72160bbf4a03f264a391b20fa52b9e87fc9d59a4d028734e
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PAPER 3 — FINAL TARGET EVALUATION AND CONFIRMATORY ANALYSIS
Amendment A2 verification: PASS
Source-model and source-stability chain: PASS
All 180 sealed target predictions verified before label access: PASS
Target label rows read so far: 0
Amendment A3 confirmatory-analysis seal SHA-256: dd48b05318d8369ece3a98ca341774c30d7521e2aa655f1742271b104fca148b
Amendment A3 verification: PASS
Target-label reveal is now authorized by the frozen A2→prediction→A3 chain.
Target label vaults opened after A3 freeze: PASS

SOURCE DATASET: NF_BoT_IoT_v3
  NF_BoT_IoT_v3: 45 target jobs evaluated — PASS

SOURCE DATASET: NF_CSE_CIC_IDS2018_v3
  NF_CSE_CIC_IDS2018_v3: 45 target jobs evaluated — PASS

SOURCE DATASET: NF_ToN_IoT_v3
  NF_ToN_IoT_v3: 45 target jobs evaluated — PASS

SOURCE DATASET: NF_U